# Fog density and object detection

**Question:** does adding a fog-density estimate help detect objects in fog, while hurting clear-image accuracy?

Compare YOLO11n, YOLO11m and DETR, each with RGB control and predicted density. Detection means finding **labeled objects and their bounding boxes**, not merely classifying an image as foggy.

The existing training profiles stay independent. `train` resumes that profile; `analyze` redraws saved results; **`evaluate` uses existing trained weights on matched clear/fog images without retraining**. This is one standalone notebook; all implementation is ordinary Python inside collapsed notebook cells.

## 1. Choose your profile and action

Change `TRAINING_PROFILE` and `SETTINGS["action"]` below. Versions live inside each profile; setup remains disposable and overwrites its fixed folder.

For the hypothesis, choose **evaluate** and point `evaluation.cityscapes_root` at a folder containing `leftImg8bit/`, `leftImg8bit_foggy/` and `gtFine/`, each with `train/` and `val/` city subfolders. The fog archive alone has no detection labels. Existing checkpoint folders are kept intact; new metrics go to `evaluation.output_name`. Change that name for a new evaluation experiment.

`source_run=""` uses the selected profile's existing run. To evaluate another saved run, enter its absolute Drive path and match its model list, seeds and image size. Oracle density is available only when a matching oracle detector was actually trained; it cannot reuse a predicted-density adapter.

**Beta** is the synthetic fog attenuation coefficient, in m⁻¹: 0 is clear; .005, .01 and .02 are increasingly foggy. Predicted density estimates beta from the image. nuScenes has unknown beta, so its normal-use results must not be called a verified clear benchmark.

**Normal rerun (`retrain=False`):** an unfinished run resumes with its recorded settings; a completed/early-stopped run redraws saved results without loading images or weights. New notebook defaults do not alter an existing experiment. **`retrain=True`** starts fresh in a separately named folder. To resume that fresh run later, set `output.run_name` to its printed folder name and switch retrain back to False. Setup remains disposable and overwrites its fixed folder.

`analyze` is valid in this notebook and only reads saved reports. After changing action/profile, rerun the settings and configuration cells (or Run all). The execution cell prints its active action and output path.


## Fog-density predictor used by detection

The object-detection study is the purpose of this notebook. The `predicted_density` arm calls a frozen model that estimates one fog beta value from each image, then passes that value to the detector adapter. It does not train the density model. Partial profiles use the completed full-run checkpoint `colab-full-study/density-42/best.pt`; set `SETTINGS["density_predictor"]["checkpoint"]` only if the checkpoint has moved. The RGB control receives zero beta, so the comparison isolates the effect of the predictor input.

In [ ]:
# fmt: off
#@title Shared constants and profile settings (expand to edit versions)
# fmt: on
from copy import deepcopy
from pathlib import Path
import os
import subprocess
import sys

# Shared names and defaults. Rename artifacts or models here once.
NOTEBOOK_REVISION = "standalone-20261012"
ENGINE_VERSION = 5  # Existing checkpoint payload format.
DEFAULT_SEED = 42
FULL_RUN_SEEDS = [DEFAULT_SEED, DEFAULT_SEED + 1, DEFAULT_SEED + 2]
YOLO_SMALL, YOLO_MEDIUM, DETR_MODEL = "YOLO11n", "YOLO11m", "DETR"
MODEL_SPECS = {
    YOLO_SMALL: "yolo11n.pt",
    YOLO_MEDIUM: "yolo11m.pt",
    DETR_MODEL: "facebook/detr-resnet-50",
}
ARM_RGB, ARM_DENSITY, ARM_ORACLE = "rgb_control", "predicted_density", "oracle_density"
DETECTION_ARMS = [ARM_RGB, ARM_DENSITY]
ARMS = [*DETECTION_ARMS, ARM_ORACLE]
ACTION_TRAIN, ACTION_PREFLIGHT, ACTION_ANALYZE, ACTION_EVALUATE = (
    "train",
    "preflight",
    "analyze",
    "evaluate",
)
PROFILE_SETUP, PROFILE_2K, PROFILE_5K, PROFILE_FULL = (
    "setup",
    "partial-2k",
    "partial-5k",
    "full",
)
PROFILE_NAMES = (PROFILE_SETUP, PROFILE_2K, PROFILE_5K, PROFILE_FULL)
SETUP_RUN_NAME, FULL_RUN_NAME = "setup-check-1", "colab-full-study"
DENSITY_RUN_PREFIX = "density"
DATA_FOLDER = "fog-study"
DRIVE_MOUNT, DRIVE_SUBDIRECTORY = "/content/drive", "MyDrive"
DATA_ARCHIVE = "fog-colab-data.zip"
MANIFEST_PATHS = {
    "manifest": "data/fog/nuscenes_all.json",
    "city_manifest": "data/fog/cityscapes.json",
}
CLASSES = ["person", "car", "truck", "bus", "train", "motorcycle", "bicycle"]
COCO_IDS, YOLO_IDS = [1, 3, 8, 6, 7, 4, 2], [0, 2, 7, 5, 6, 3, 1]
BETA_SCALE = 0.02
DENSITY_INPUT_SIZE = 128
BENCHMARK_IMAGES, BENCHMARK_BATCH_SIZE = 2000, 64
DENSITY_BIN_NAMES = ("clear", "light", "moderate", "dense")
DENSITY_BIN_UPPER_BOUNDS = (0.0, 0.005, 0.01, BETA_SCALE)
PROFILE_SELECTION_SCHEMA = 1
VALID_EVALUATION_SPLITS = {"val", "test"}
BEST_CHECKPOINT, LAST_CHECKPOINT = "best.pt", "last.pt"
RUN_SETTINGS_FILE, INPUT_MANIFEST_FILE = "settings.json", "input-manifest.json"
INPUT_SELECTION_FILE, SUBSET_AUDIT_FILE = (
    "input-selection.json",
    "subset-selection.json",
)
PROGRESS_FILE, LOSS_HISTORY_FILE = "progress.json", "history.csv"
RESULTS_FILE, STATUS_FILE, STATUS_TABLE_FILE = (
    "results.csv",
    "status.json",
    "status.csv",
)
PAIRED_DELTA_FILE, DENSITY_REPORT_FILE = "paired-deltas.csv", "density-quality.csv"
MODEL_EXPORT_FILE, RUN_LOCK_FILE, FIGURE_DIRECTORY = (
    "trained-model.pt",
    "run.lock",
    "figures",
)
SEED_SUMMARY_FILE, HYPOTHESIS_FILE = "seed-summary.csv", "fog-vs-clear.csv"
MATCHED_MANIFEST_FILE = "matched-cityscapes.json"
TRAINING_DEFAULTS = {
    "density_epochs": 15,
    "density_checkpoint": None,
    "density_sampling": "natural",
    "batch_size": 1,
    "effective_batch_size": 4,
    "accumulate": 4,
    "loader": {"num_workers": 0, "prefetch_factor": None, "pin_memory": False},
    "repeats": 1,
    "learning_rates": {YOLO_SMALL: 1e-4, YOLO_MEDIUM: 1e-4, DETR_MODEL: 1e-5},
    "adapter_lr": 1e-3,
}
FULL_RUN_DEFAULTS = {
    key: TRAINING_DEFAULTS[key]
    for key in ("batch_size", "effective_batch_size", "loader", "density_sampling")
}
FULL_RUN_IMMUTABLE_KEYS = tuple(TRAINING_DEFAULTS) + (
    "epochs",
    "seeds",
    "arms",
    "size",
    "save_every",
    "amp",
    "device",
    "checkpoints",
)
PROFILE_COMMON = {
    "seed": DEFAULT_SEED,
    "models": list(MODEL_SPECS),
    "size": 640,
    "save_every": 200,
    "evaluation_split": "val",
    "early_stopping_patience": 2,
    "overwrite_existing": False,
    "density_sampling": "natural",
}
PROFILE_SETTINGS = {
    PROFILE_SETUP: {
        **PROFILE_COMMON,
        "training_image_cap": 48,
        "tuning_image_cap": 24,
        "epochs": 1,
        "size": 320,
        "save_every": 2,
        "early_stopping_patience": 0,
        "overwrite_existing": True,
        "run_name": SETUP_RUN_NAME,
    },
    PROFILE_2K: {
        **PROFILE_COMMON,
        "training_image_cap": 2000,
        "tuning_image_cap": 500,
        "epochs": 5,
        "version": "v2",
        "run_name": PROFILE_2K,
    },
    PROFILE_5K: {
        **PROFILE_COMMON,
        "training_image_cap": 5000,
        "tuning_image_cap": 500,
        "epochs": 5,
        "version": "v3",
        "run_name": PROFILE_5K,
    },
    PROFILE_FULL: {
        **PROFILE_COMMON,
        "training_image_cap": None,
        "tuning_image_cap": None,
        "epochs": 30,
        "evaluation_split": "test",
        "early_stopping_patience": 0,
        "version": "",
        "run_name": FULL_RUN_NAME,
    },
}
FOG_DENSITY_MODEL = None
FOG_DENSITY_MODEL_KEY = None

### Run settings

Choose the profile and action here. Versions belong to each entry in `PROFILE_SETTINGS` above. The predictor checkpoint is also configured here; the **Load the fog-density model** step below actually opens it and freezes its weights.

In [ ]:
# Edit the profile version above and these run settings.
RUN_ENVIRONMENT = "colab"  # colab | local
TRAINING_PROFILE = PROFILE_2K
LOCAL_ROOT = Path.home() / DATA_FOLDER
ENVIRONMENT_PROFILES = {
    "colab": {
        "runtime_root": f"/content/{DATA_FOLDER}",
        "data_location": {"kind": "drive", "path": DATA_FOLDER},
        "output_location": {"kind": "drive", "path": f"{DATA_FOLDER}/runs"},
        "data_mode": "zip",
        "low_disk": True,
        "device": "cuda:0",
        "amp": True,
    },
    "local": {
        "runtime_root": str(LOCAL_ROOT / "runtime"),
        "data_location": {"kind": "path", "path": str(LOCAL_ROOT / "input")},
        "output_location": {"kind": "path", "path": str(LOCAL_ROOT / "runs")},
        "data_mode": "zip",
        "low_disk": False,
        "device": "cpu",
        "amp": False,
    },
}
if (
    RUN_ENVIRONMENT not in ENVIRONMENT_PROFILES
    or TRAINING_PROFILE not in PROFILE_SETTINGS
):
    raise ValueError("Choose a listed environment and training profile.")
ENVIRONMENT = deepcopy(ENVIRONMENT_PROFILES[RUN_ENVIRONMENT])
SELECTED_PROFILE = deepcopy(PROFILE_SETTINGS[TRAINING_PROFILE])
version = (
    SELECTED_PROFILE.get("version", "") if TRAINING_PROFILE != PROFILE_SETUP else ""
)
run_name = SELECTED_PROFILE["run_name"] + (f"-{version}" if version else "")
SELECTED_PROFILE.update(
    name=TRAINING_PROFILE,
    run_name=run_name,
    run_suffix=f"-{run_name}",
    initialization="existing_full_run"
    if TRAINING_PROFILE == PROFILE_FULL
    else "pretrained",
)
SETTINGS = {
    "action": ACTION_TRAIN,  # train | preflight | analyze | evaluate
    "retrain": False,
    "profile": SELECTED_PROFILE,
    "density_predictor": {
        "checkpoint": "",
        "source_run": FULL_RUN_NAME,
        "seed": DEFAULT_SEED,
    },
    "evaluation": {
        "source_run": "",
        "output_name": run_name + "-matched-evaluation",
        "cityscapes_root": str(
            Path(DRIVE_MOUNT) / DRIVE_SUBDIRECTORY / DATA_FOLDER / "cityscapes-labeled"
        ),
        "split": "test",
        "max_pairs": 100,
    },
    "test": {
        "enabled": False,
        "models": [YOLO_SMALL],
        "seed": DEFAULT_SEED,
        "epochs": 1,
        "density_epochs": 1,
        "image_size": 320,
    },
    "runtime_root": ENVIRONMENT["runtime_root"],
    "drive_mount": DRIVE_MOUNT,
    "drive_subdir": DRIVE_SUBDIRECTORY,
    "data": {
        "location": ENVIRONMENT["data_location"],
        "mode": ENVIRONMENT["data_mode"],
        "archive": DATA_ARCHIVE,
        "url": "",
        "low_disk": ENVIRONMENT["low_disk"],
        "minimum_free_gib": 15,
        "manifests": deepcopy(MANIFEST_PATHS),
        "city_images": "data/fog/cityscapes",
        "nuscenes": "data/depth/datasets/nuscenes",
        "allowed_archive_roots": ["data/", ".cache/"],
        "extraction_marker": ".data-extracted.json",
    },
    "output": {
        "location": ENVIRONMENT["output_location"],
        "run_name": run_name,
        "export_name": run_name + "-results",
        "packages_file": "colab-packages.txt",
    },
    "models": deepcopy(MODEL_SPECS),
    "training": {
        **deepcopy(TRAINING_DEFAULTS),
        "epochs": SELECTED_PROFILE["epochs"],
        "seeds": [SELECTED_PROFILE["seed"]]
        if TRAINING_PROFILE != PROFILE_FULL
        else list(FULL_RUN_SEEDS),
        "arms": list(DETECTION_ARMS),
        "size": SELECTED_PROFILE["size"],
        "save_every": SELECTED_PROFILE["save_every"],
        "amp": ENVIRONMENT["amp"],
        "device": ENVIRONMENT["device"],
    },
}
print(
    f"Profile: {TRAINING_PROFILE} | version: {version or 'base'} | output: {run_name}"
)
print(
    f"Device: {SETTINGS['training']['device']} | models: {', '.join(SELECTED_PROFILE['models'])}"
)

### Connect Google Drive before setup

Authorize Drive now, before dependency installation. Local filesystem profiles skip this step.


In [ ]:
# Authenticate before package installation or other long-running setup.
if any(
    SETTINGS[section]["location"]["kind"] == "drive" for section in ("data", "output")
):
    from google.colab import drive

    drive.mount(SETTINGS["drive_mount"])
    drive_root = Path(SETTINGS["drive_mount"]) / SETTINGS["drive_subdir"]
    if not drive_root.is_dir():
        raise RuntimeError("Drive is unavailable. Rerun this connection cell.")
    print("Google Drive connected. Continuing setup.")
else:
    print("Using local filesystem paths; no Drive connection needed.")

## 2. Set up the runtime

Choose storage, create the run folders, install dependencies, and check the GPU. All study implementation stays in this notebook.

In [ ]:
ROOT = Path(SETTINGS["runtime_root"])
if not ROOT.is_absolute():
    raise ValueError("runtime_root must be an absolute path.")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

### Install packages and check the GPU

In [ ]:
REQUIRED_PACKAGES = [
    "matplotlib",
    "numpy",
    "pandas",
    "pillow",
    "ijson",
    "shapely",
    "pyquaternion",
    "scipy",
    "filelock",
    "tqdm",
    "ultralytics==8.4.155",
    "transformers==5.17.0",
    "pycocotools==2.0.11",
]
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--only-binary=:all:",
        *REQUIRED_PACKAGES,
    ]
)

### Supporting functions and classes (collapsed)

All implementation lives below, organized by purpose. These cells define the functions used by the visible steps; they do not run training or create Python files.

#### Detector models and detection scoring

In [ ]:
# fmt: off
#@title Supporting code: Detector models and detection scoring
# fmt: on
import contextlib
import hashlib
import io
import json
import random
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.nn import functional as F
import os
import sqlite3
from collections import Counter
import ijson
from filelock import FileLock
from pyquaternion import Quaternion
from shapely.geometry import MultiPoint, box as rectangle
from tqdm.auto import tqdm
from dataclasses import dataclass, field
import gc
import importlib.metadata
import shutil
import stat
import urllib.request
import zipfile
from uuid import uuid4
from pathlib import Path, PurePosixPath
from IPython.display import Image as PreviewImage, display

"""Detector, density-conditioning and scoring primitives for the fog study.

The visible train step fine-tunes the detector and adapter together.
"""


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def density_bin(beta):
    """Map a known beta to a stable, interpretable fog-density stratum."""
    if beta is None:
        return "unknown"
    value = float(beta)
    for name, upper_bound in zip(DENSITY_BIN_NAMES, DENSITY_BIN_UPPER_BOUNDS):
        if value <= upper_bound:
            return name
    raise ValueError(f"Beta is outside the supported range: {value}")


def epoch_indices(rows, seed, epoch, balanced=False):
    """Return a deterministic epoch order, optionally equalizing known beta strata."""
    if not rows:
        return []
    groups = {}
    for index, row in enumerate(rows):
        label = density_bin(row.get("beta")) if balanced else "all"
        groups.setdefault(label, []).append(index)
    generator = random.Random(seed + epoch)
    if not balanced:
        return generator.sample(range(len(rows)), len(rows))
    target_size = max(len(indices) for indices in groups.values())
    order = []
    for label in sorted(groups):
        indices = groups[label]
        order.extend(generator.choices(indices, k=target_size))
    generator.shuffle(order)
    return order


def epoch_size(rows, balanced=False):
    """Return the number of image steps represented by one configured epoch."""
    if not rows or not balanced:
        return len(rows)
    counts = {}
    for row in rows:
        label = density_bin(row.get("beta"))
        counts[label] = counts.get(label, 0) + 1
    return max(counts.values()) * len(counts)


def digest(path):
    result = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            result.update(chunk)
    return result.hexdigest()


def read_manifest(path, verify_images=False):
    """Fail closed on incomplete datasets; never turn missing annotations into empty GT."""
    path = Path(path)
    records = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(records, list) or not records:
        raise ValueError("Manifest must contain a nonempty list of labeled images.")
    ids, images, groups = set(), set(), {}
    for row in records:
        for key in (
            "id",
            "image",
            "dataset",
            "split",
            "group",
            "boxes",
            "labels",
            "beta",
        ):
            if key not in row:
                raise ValueError(f"Missing {key} in manifest row")
        if row["dataset"] not in ("foggy_cityscapes", "nuscenes"):
            raise ValueError("Unknown dataset")
        if row["split"] not in ("density_train", "detector_train", "val", "test"):
            raise ValueError("Invalid split")
        if not row["id"] or not row["group"]:
            raise ValueError("Empty image/group identity")
        image = (path.parent / row["image"]).resolve()
        if row["id"] in ids or str(image) in images:
            raise ValueError("Duplicate image identity/path")
        ids.add(row["id"])
        images.add(str(image))
        if not image.is_file():
            raise FileNotFoundError(image)
        row["image"] = str(image)
        if verify_images or "width" not in row or "height" not in row:
            with Image.open(image) as img:
                width, height = img.size
                if verify_images:
                    img.verify()
        else:
            width, height = row["width"], row["height"]
        if width <= 0 or height <= 0:
            raise ValueError("Invalid image dimensions")
        boxes = np.asarray(row["boxes"], dtype=float).reshape(-1, 4)
        labels = np.asarray(row["labels"])
        if len(boxes) != len(labels) or not np.isfinite(boxes).all():
            raise ValueError("Invalid boxes/labels")
        crowds = row.get("iscrowd", [0] * len(boxes))
        if len(crowds) != len(boxes) or any(c not in (0, 1) for c in crowds):
            raise ValueError("iscrowd must have one binary flag per annotation")
        if len(labels) and (not np.isin(labels, range(len(CLASSES))).all()):
            raise ValueError("Labels must use shared class IDs 0..6")
        if len(boxes) and (
            np.any(boxes[:, 2:] <= boxes[:, :2])
            or np.any(boxes < 0)
            or np.any(boxes > [width, height, width, height])
        ):
            raise ValueError("Boxes must be positive-area XYXY pixels inside the image")
        beta = row["beta"]
        if beta is not None and (not np.isfinite(beta) or not 0 <= beta <= BETA_SCALE):
            raise ValueError(
                "Beta must be null or an attenuation coefficient in [0, .02] m^-1"
            )
        if row["dataset"] == "foggy_cityscapes" and beta is None:
            raise ValueError("Foggy Cityscapes requires known beta")
        key = (row["dataset"], row["group"])
        if key in groups and groups[key] != row["split"]:
            raise ValueError("Scene/sequence leakage across splits")
        groups[key] = row["split"]
        row["width"], row["height"] = width, height
    return records


def select_groups(rows, limit=None):
    """Select whole sequences, preserving all fog variants; None selects everything."""
    keys = sorted({(r["dataset"], r["group"]) for r in rows})
    if limit is not None:
        if limit < 1:
            raise ValueError("Group limit must be positive or None")
        keys = keys[:limit]
    keys = set(keys)
    return sorted(
        [r for r in rows if (r["dataset"], r["group"]) in keys], key=lambda r: r["id"]
    )


def _stable_order(values, seed, label):
    return sorted(
        values,
        key=lambda value: hashlib.sha256(
            f"{PROFILE_SELECTION_SCHEMA}:{seed}:{label}:{value}".encode()
        ).hexdigest(),
    )


def select_scene_safe_rows(rows, image_cap, seed, split, label):
    """Select complete scene groups without crossing split boundaries."""
    candidates = [row for row in rows if row["split"] == split]
    if image_cap is None:
        return sorted(candidates, key=lambda row: row["id"])
    if not isinstance(image_cap, int) or image_cap < 1:
        raise ValueError("Profile image caps must be positive integers or None")
    groups = {}
    for row in candidates:
        groups.setdefault((row["dataset"], row["group"]), []).append(row)
    selected, count = [], 0
    for group in _stable_order(groups, seed, label):
        group_rows = sorted(groups[group], key=lambda row: row["id"])
        if count and count + len(group_rows) > image_cap:
            continue
        selected.extend(group_rows)
        count += len(group_rows)
        if count >= image_cap:
            break
    if not selected and candidates:
        smallest = min(groups.values(), key=len)
        selected = sorted(smallest, key=lambda row: row["id"])
    return sorted(selected, key=lambda row: row["id"])


def select_profile_rows(rows, profile, seed, limits=None):
    """Build deterministic, scene-safe rows for one independent profile."""
    if profile not in PROFILE_NAMES:
        raise ValueError(f"Unknown training profile: {profile}")
    if profile == PROFILE_FULL:
        return sorted(rows, key=lambda row: row["id"])
    profile_limits = {
        PROFILE_SETUP: {"detector_train": 48, "density_train": 48, "val": 24},
        PROFILE_2K: {"detector_train": 2000, "density_train": 1000, "val": 200},
        PROFILE_5K: {"detector_train": 5000, "density_train": 2000, "val": 500},
    }[profile]
    limits = limits or profile_limits
    selected = []
    for split in ("detector_train", "density_train", "val"):
        if limits is not None and split not in limits:
            continue
        label = "tuning" if split == "val" else f"{profile}:{split}"
        selected.extend(select_scene_safe_rows(rows, limits[split], seed, split, label))
    return sorted(selected, key=lambda row: row["id"])


def image_tensor(row, size, device):
    with Image.open(row["image"]) as image:
        image = image.convert("RGB").resize((size, size), Image.Resampling.BILINEAR)
        array = np.asarray(image).copy()
    return (
        torch.from_numpy(array).permute(2, 0, 1).float().unsqueeze(0).to(device) / 255
    )


class FogAdapter(nn.Module):
    """Identity-initialized RGB residual with scalar FiLM at hidden features.

    The control has exactly the same parameters, but always receives zero.
    This tests input adaptation, not internal YOLO/DETR feature conditioning.
    """

    def __init__(self):
        super().__init__()
        self.features = nn.Conv2d(3, 8, 3, padding=1)
        self.film = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 16))
        self.output = nn.Conv2d(8, 3, 1)
        nn.init.zeros_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, image, beta):
        scale, shift = self.film(beta.reshape(-1, 1) / BETA_SCALE).chunk(2, dim=1)
        features = self.features(image)
        features = features * (1 + scale[:, :, None, None]) + shift[:, :, None, None]
        return (image + 0.1 * self.output(F.relu(features)).tanh()).clamp(0, 1)


class Detector:
    """Differentiable detector loss plus consistent, original-pixel predictions."""

    def __init__(self, name, checkpoint, device, trainable=True):
        self.name, self.device = name, device
        self.yolo = name in (YOLO_SMALL, YOLO_MEDIUM)
        if self.yolo:
            # Ultralytics selects compiled NMS only when torchvision is loaded.
            # Load it before warm-up/timing to avoid the slow Python fallback.
            import torchvision.ops  # noqa: F401 - import enables compiled NMS
            from ultralytics import YOLO
            from ultralytics.utils import DEFAULT_CFG_DICT

            self.model = YOLO(str(checkpoint)).model.to(device).float()
            self.model.args = SimpleNamespace(**DEFAULT_CFG_DICT)
        else:
            from transformers import DetrConfig, DetrForObjectDetection

            kwargs = {} if Path(checkpoint).is_dir() else {"revision": "no_timm"}
            raw, _ = DetrConfig.get_config_dict(str(checkpoint), **kwargs)
            # The official no_timm export predates strict bool validation in v5.
            if raw.get("dilation") is None:
                raw["dilation"] = False
            if raw.get("use_pretrained_backbone") is None:
                raw["use_pretrained_backbone"] = False
            configuration = DetrConfig.from_dict(raw)
            self.model = DetrForObjectDetection.from_pretrained(
                str(checkpoint), config=configuration, **kwargs
            ).to(device)
        self.model.eval().requires_grad_(trainable)
        if self.yolo:
            # YOLO's fixed distribution-to-distance integral is not a learned convolution.
            for name, parameter in self.model.named_parameters():
                if ".dfl." in name:
                    parameter.requires_grad_(False)

    def training_mode(self):
        self.model.train()
        # Batch-one microsteps: preserve pretrained BN statistics, but train affine weights.
        for module in self.model.modules():
            if isinstance(module, nn.modules.batchnorm._BatchNorm):
                module.eval()

    def forward(self, x):
        if self.yolo:
            return self.model(x)
        mean = x.new_tensor([0.485, 0.456, 0.406])[None, :, None, None]
        std = x.new_tensor([0.229, 0.224, 0.225])[None, :, None, None]
        return self.model(pixel_values=(x - mean) / std)

    def loss(self, x, row):
        boxes = x.new_tensor(row["boxes"]).reshape(-1, 4)
        boxes = boxes / x.new_tensor([row["width"], row["height"]] * 2)
        center = (boxes[:, :2] + boxes[:, 2:]) / 2
        wh = boxes[:, 2:] - boxes[:, :2]
        boxes = torch.cat([center, wh], dim=1)
        mapping = YOLO_IDS if self.yolo else COCO_IDS
        labels = torch.tensor([mapping[i] for i in row["labels"]], device=self.device)
        if self.yolo:
            batch = {
                "img": x,
                "batch_idx": x.new_zeros(len(labels)),
                "cls": labels.reshape(-1, 1),
                "bboxes": boxes,
            }
            return self.model.loss(batch, self.forward(x))[0].sum()
        mean = x.new_tensor([0.485, 0.456, 0.406])[None, :, None, None]
        std = x.new_tensor([0.229, 0.224, 0.225])[None, :, None, None]
        return self.model(
            pixel_values=(x - mean) / std,
            labels=[{"class_labels": labels.long(), "boxes": boxes}],
        ).loss

    def predict(self, x, row):
        output = self.forward(x)
        if self.yolo:
            from ultralytics.utils.nms import non_max_suppression

            result = non_max_suppression(
                output, conf_thres=0.001, iou_thres=0.7, classes=YOLO_IDS, max_det=100
            )[0]
            boxes, scores, labels = result[:, :4], result[:, 4], result[:, 5].long()
            boxes = boxes / x.shape[-1]
            mapping = YOLO_IDS
        else:
            probabilities = output.logits.softmax(-1)[0, :, :-1]
            scores, labels = probabilities.max(-1)
            cxcy, wh = output.pred_boxes[0].split(2, dim=-1)
            boxes = torch.cat([cxcy - wh / 2, cxcy + wh / 2], dim=-1)
            mapping = COCO_IDS
        boxes = boxes.clamp(0, 1) * x.new_tensor([row["width"], row["height"]] * 2)
        result = []
        for box, score, label in zip(
            boxes.detach().cpu().tolist(),
            scores.detach().cpu().tolist(),
            labels.detach().cpu().tolist(),
        ):
            if (
                label in mapping
                and score >= 0.001
                and box[2] > box[0]
                and box[3] > box[1]
            ):
                result.append(
                    {"box": box, "score": score, "label": mapping.index(label)}
                )
        return sorted(result, key=lambda r: -r["score"])[:100]


def beta_for(arm, x, row, density):
    if arm == ARM_RGB:
        return x.new_zeros((1, 1))
    if arm == ARM_DENSITY:
        return get_fog_density(x, density)
    if arm == ARM_ORACLE and row["beta"] is not None:
        return x.new_tensor([[row["beta"]]])
    raise ValueError(
        "Oracle density is unavailable; do not substitute zero or a prediction"
    )


def coco_metrics(rows, results, include_curve=False):
    """COCO bbox AP, including crowd annotations; seven shared categories."""
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    if [r["id"] for r in rows] != [r["id"] for r in results]:
        raise ValueError("Predictions are not paired with the exact evaluation images")
    images, annotations, predictions = [], [], []
    for index, (row, result) in enumerate(zip(rows, results), 1):
        images.append({"id": index, "width": row["width"], "height": row["height"]})
        crowds = row.get("iscrowd", [0] * len(row["boxes"]))
        for box, label, crowd in zip(row["boxes"], row["labels"], crowds):
            x1, y1, x2, y2 = box
            annotations.append(
                {
                    "id": len(annotations) + 1,
                    "image_id": index,
                    "category_id": int(label),
                    "bbox": [x1, y1, x2 - x1, y2 - y1],
                    "area": (x2 - x1) * (y2 - y1),
                    "iscrowd": int(crowd),
                }
            )
        for pred in result["prediction"]:
            x1, y1, x2, y2 = pred["box"]
            predictions.append(
                {
                    "image_id": index,
                    "category_id": pred["label"],
                    "bbox": [x1, y1, x2 - x1, y2 - y1],
                    "score": pred["score"],
                }
            )
    truth = COCO()
    truth.dataset = {
        "info": {},
        "images": images,
        "annotations": annotations,
        "categories": [{"id": i, "name": n} for i, n in enumerate(CLASSES)],
    }
    with contextlib.redirect_stdout(io.StringIO()):
        truth.createIndex()
        if predictions:
            predicted = truth.loadRes(predictions)
        else:
            predicted = COCO()
            predicted.dataset = {**truth.dataset, "annotations": []}
            predicted.createIndex()
        evaluator = COCOeval(truth, predicted, "bbox")
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
    tp = fp = positives = 0
    for item in evaluator.evalImgs:
        if item is None or item["aRng"] != evaluator.params.areaRng[0]:
            continue
        keep = np.asarray(item["dtScores"]) >= 0.25
        ignored = np.asarray(item["dtIgnore"])[0].astype(bool)
        matched = np.asarray(item["dtMatches"])[0] > 0
        tp += int(np.sum(keep & ~ignored & matched))
        fp += int(np.sum(keep & ~ignored & ~matched))
        positives += int(np.sum(~np.asarray(item["gtIgnore"]).astype(bool)))
    metrics = {
        "mAP50_95": float(evaluator.stats[0]),
        "AP50": float(evaluator.stats[1]),
        "AR100": float(evaluator.stats[8]),
        "precision_at_025": tp / (tp + fp) if tp + fp else 0.0,
        "recall_at_025": tp / positives if positives else np.nan,
    }
    for index, name in enumerate(CLASSES):
        values = evaluator.eval["precision"][:, :, index, 0, -1]
        supported = values[values >= 0]
        metrics[f"AP_{name}"] = float(supported.mean()) if len(supported) else np.nan
    metrics = {key: np.nan if value == -1 else value for key, value in metrics.items()}
    if include_curve:
        precision = evaluator.eval["precision"][0, :, :, 0, -1]
        valid = precision >= 0
        counts = valid.sum(axis=1)
        means = np.divide(
            np.where(valid, precision, 0).sum(axis=1),
            counts,
            out=np.full(len(counts), np.nan),
            where=counts > 0,
        )
        metrics["precision_curve"] = means.tolist()
        metrics["recall_curve"] = evaluator.params.recThrs.tolist()
    return metrics


def summarize(rows, results):
    groups = [("all", "all", list(range(len(rows))))]
    fog_indices = [
        i for i, row in enumerate(rows) if row["beta"] is not None and row["beta"] > 0
    ]
    if fog_indices:
        groups.append(("foggy", "foggy", fog_indices))
    for beta in sorted({r["beta"] for r in rows if r["beta"] is not None}):
        groups.append(
            (
                str(beta),
                density_bin(beta),
                [i for i, r in enumerate(rows) if r["beta"] == beta],
            )
        )
    if any(r["beta"] is None for r in rows):
        groups.append(
            (
                "unknown",
                "unknown",
                [i for i, r in enumerate(rows) if r["beta"] is None],
            )
        )
    summaries = []
    for bucket, bin_name, indices in groups:
        subset = [rows[i] for i in indices]
        outputs = [results[i] for i in indices]
        values = [t["total_ms"] for r in outputs for t in r["timings"]]
        metric = coco_metrics(subset, outputs)
        summaries.append(
            {
                "density": bucket,
                "density_bin": bin_name,
                "condition": "clear"
                if bin_name == "clear"
                else "foggy"
                if bin_name in ("light", "moderate", "dense", "foggy")
                else "unknown"
                if bin_name == "unknown"
                else "mixed",
                "distribution": "natural",
                "images": len(subset),
                **metric,
                "mean_ms": float(np.mean(values)),
                "p50_ms": float(np.median(values)),
                "p95_ms": float(np.percentile(values, 95)),
                "serial_fps": 1000 / float(np.mean(values)),
                "density_mean_ms": float(
                    np.mean([t["density_ms"] for r in outputs for t in r["timings"]])
                ),
            }
        )
    return summaries

#### nuScenes data preparation

In [ ]:
# fmt: off
#@title Supporting code: nuScenes data preparation
# fmt: on
"""Resumable nuScenes preparation without loading millions of sweep records into RAM."""


CAMERAS = (
    "CAM_FRONT",
    "CAM_FRONT_RIGHT",
    "CAM_BACK_RIGHT",
    "CAM_BACK",
    "CAM_BACK_LEFT",
    "CAM_FRONT_LEFT",
)
SCHEMA = 3


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)


def stream(path):
    with Path(path).open("rb") as handle:
        yield from ijson.items(handle, "item", use_float=True)


def source_signature(root, version):
    names = (
        "scene",
        "sample",
        "sample_data",
        "sample_annotation",
        "ego_pose",
        "calibrated_sensor",
        "sensor",
        "instance",
        "category",
    )
    return {
        "schema": SCHEMA,
        "root": str(root),
        "version": version,
        "tables": {
            name: [
                ((root / version) / f"{name}.json").stat().st_size,
                ((root / version) / f"{name}.json").stat().st_mtime_ns,
            ]
            for name in names
        },
    }


def metadata_index(root, version, cache):
    signature = source_signature(root, version)
    db_path = cache / "metadata.sqlite"
    stamp = cache / "metadata-source.json"
    if (
        db_path.is_file()
        and stamp.is_file()
        and json.loads(stamp.read_text()) == signature
    ):
        return db_path
    temp = cache / "metadata-building.sqlite"
    # Recreate only this explicitly named disposable database after an interrupted import.
    if temp.exists():
        temp.unlink()
    connection = sqlite3.connect(temp)
    try:
        connection.execute("PRAGMA journal_mode=OFF")
        connection.execute("PRAGMA synchronous=OFF")
        connection.execute(
            "CREATE TABLE records(kind TEXT, token TEXT, parent TEXT, payload TEXT, PRIMARY KEY(kind,token))"
        )
        camera_poses = set()
        for kind in signature["tables"]:
            batch = []
            for row in tqdm(
                stream(root / version / f"{kind}.json"),
                desc=f"Index {kind}",
                unit=" rows",
                mininterval=2,
            ):
                if kind == "sample_data":
                    if not row["is_key_frame"] or not row["filename"].startswith(
                        "samples/CAM_"
                    ):
                        continue
                    camera_poses.add(row["ego_pose_token"])
                if kind == "ego_pose" and row["token"] not in camera_poses:
                    continue
                parent = row.get("sample_token", row.get("scene_token", ""))
                batch.append((kind, row["token"], parent, json.dumps(row)))
                if len(batch) >= 5000:
                    connection.executemany("INSERT INTO records VALUES(?,?,?,?)", batch)
                    batch.clear()
            if batch:
                connection.executemany("INSERT INTO records VALUES(?,?,?,?)", batch)
            connection.commit()
        connection.execute("CREATE INDEX by_parent ON records(kind,parent)")
        connection.commit()
    finally:
        connection.close()
    os.replace(temp, db_path)
    atomic_json(stamp, signature)
    return db_path


def clipped_bounds(points, width, height):
    """Clip the projected convex hull, not its bounding rectangle (not equivalent)."""
    if len(points) < 3 or not np.isfinite(points).all():
        return None
    lo, hi = points.min(axis=0), points.max(axis=0)
    if hi[0] <= 0 or hi[1] <= 0 or lo[0] >= width or lo[1] >= height:
        return None
    hull = MultiPoint(points).convex_hull.intersection(rectangle(0, 0, width, height))
    return list(hull.bounds) if not hull.is_empty and hull.area > 0 else None


def prepare_nuscenes(root, output, version="v1.0-trainval"):
    """Build complete scene shards with progress; resume after interruption.

    Official train: deterministic 90% train / 10% tuning by scene. Official val:
    held-out test. Every labeled keyframe and all six cameras are accounted for.
    Missing files are recorded and block a full-data claim unless explicitly allowed.
    """
    from nuscenes.utils.splits import create_splits_scenes
    from nuscenes.utils.data_classes import Box

    root, output = Path(root).resolve(), Path(output).resolve()
    cache = output.parent / (output.stem + "-preparation")
    cache.mkdir(parents=True, exist_ok=True)
    with FileLock(str(cache / "prepare.lock"), timeout=0):
        db = metadata_index(root, version, cache)
        connection = sqlite3.connect(db)
        connection.execute("PRAGMA cache_size=-64000")

        def table(kind):
            return {
                row["token"]: row
                for (payload,) in connection.execute(
                    "SELECT payload FROM records WHERE kind=?", (kind,)
                )
                for row in [json.loads(payload)]
            }

        def children(kind, parent):
            return [
                json.loads(p)
                for (p,) in connection.execute(
                    "SELECT payload FROM records WHERE kind=? AND parent=?",
                    (kind, parent),
                )
            ]

        def get(kind, token):
            return json.loads(
                connection.execute(
                    "SELECT payload FROM records WHERE kind=? AND token=?",
                    (kind, token),
                ).fetchone()[0]
            )

        sensors, calibrated = table("sensor"), table("calibrated_sensor")
        categories, instances = table("category"), table("instance")
        scenes = table("scene")
        splits = create_splits_scenes()
        train_names = set(splits["mini_train" if version == "v1.0-mini" else "train"])
        test_names = set(splits["mini_val" if version == "v1.0-mini" else "val"])
        ordered = sorted(
            [s for s in scenes.values() if s["name"] in train_names | test_names],
            key=lambda r: r["name"],
        )
        report = {
            "source": source_signature(root, version),
            "expected_scenes": len(train_names | test_names),
            "selected_scenes": len(ordered),
            "expected_images": 0,
            "available_images": 0,
            "missing_images": [],
            "missing_channels": [],
        }
        shard_paths = []
        for scene in tqdm(ordered, desc="Project scenes (resumable)", mininterval=2):
            shard = cache / f"{scene['name']}.json"
            if shard.exists():
                data = json.loads(shard.read_text(encoding="utf-8"))
                # Recompute scenes that previously had missing images: newly downloaded
                # files become eligible on the next preparation run.
                reusable = (
                    data.get("source") == report["source"]
                    and not data["missing_images"]
                )
                reusable = reusable and all(
                    Path(r["image"]).is_file() for r in data["rows"]
                )
            else:
                reusable = False
            if not reusable:
                bucket = (
                    int(hashlib.sha256(scene["name"].encode()).hexdigest()[:8], 16) % 10
                )
                split = (
                    "test"
                    if scene["name"] in test_names
                    else ("val" if bucket == 0 else "detector_train")
                )
                data = {
                    "rows": [],
                    "expected_images": 0,
                    "missing_images": [],
                    "missing_channels": [],
                    "source": report["source"],
                }
                for sample in sorted(
                    children("sample", scene["token"]), key=lambda r: r["timestamp"]
                ):
                    world_boxes = []
                    for ann in children("sample_annotation", sample["token"]):
                        category = categories[
                            instances[ann["instance_token"]]["category_token"]
                        ]["name"]
                        cls = (
                            "person"
                            if category.startswith("human.pedestrian.")
                            and category.rsplit(".", 1)[-1]
                            not in ("stroller", "wheelchair", "personal_mobility")
                            else "bus"
                            if category.startswith("vehicle.bus.")
                            else {
                                "vehicle.car": "car",
                                "vehicle.truck": "truck",
                                "vehicle.bicycle": "bicycle",
                                "vehicle.motorcycle": "motorcycle",
                            }.get(category)
                        )
                        if cls is not None:
                            box = Box(
                                ann["translation"],
                                ann["size"],
                                Quaternion(ann["rotation"]),
                            )
                            world_boxes.append((box.corners(), CLASSES.index(cls)))
                    available_channels = set()
                    for sd in children("sample_data", sample["token"]):
                        cs = calibrated[sd["calibrated_sensor_token"]]
                        channel = sensors[cs["sensor_token"]]["channel"]
                        if channel not in CAMERAS:
                            continue
                        available_channels.add(channel)
                        image = root / sd["filename"]
                        if not image.is_file():
                            data["missing_images"].append(str(image))
                            continue
                        ego = get("ego_pose", sd["ego_pose_token"])
                        rot_ego = Quaternion(ego["rotation"]).rotation_matrix.T
                        rot_cam = Quaternion(cs["rotation"]).rotation_matrix.T
                        rotation = rot_cam @ rot_ego
                        translation = -rotation @ np.array(
                            ego["translation"]
                        ) - rot_cam @ np.array(cs["translation"])
                        intrinsic = np.array(cs["camera_intrinsic"])
                        boxes, labels = [], []
                        for corners, cls in world_boxes:
                            camera = rotation @ corners + translation[:, None]
                            camera = camera[:, camera[2] > 0]
                            if camera.shape[1] < 3:
                                continue
                            projected = intrinsic @ camera
                            points = (projected[:2] / projected[2]).T
                            bounds = clipped_bounds(points, sd["width"], sd["height"])
                            if bounds is not None:
                                boxes.append(bounds)
                                labels.append(cls)
                        data["rows"].append(
                            {
                                "id": f"nuscenes/{sd['token']}",
                                "image": str(image),
                                "dataset": "nuscenes",
                                "group": scene["name"],
                                "split": split,
                                "beta": None,
                                "camera": channel,
                                "width": sd["width"],
                                "height": sd["height"],
                                "boxes": boxes,
                                "labels": labels,
                                "iscrowd": [0] * len(boxes),
                                "label_policy": "projected 3D boxes, all visibility levels; 2D proxy benchmark",
                            }
                        )
                    data["expected_images"] += len(CAMERAS)
                    data["missing_channels"].extend(
                        f"{sample['token']}/{c}"
                        for c in set(CAMERAS) - available_channels
                    )
                atomic_json(shard, data)
            report["expected_images"] += data["expected_images"]
            report["available_images"] += len(data["rows"])
            report["missing_images"].extend(data["missing_images"])
            report["missing_channels"].extend(data["missing_channels"])
            shard_paths.append(shard)
            if len(shard_paths) % 25 == 0:
                atomic_json(output.with_suffix(".coverage.json"), report)
        connection.close()
        complete = (
            len(ordered) == len(train_names | test_names)
            and not report["missing_images"]
            and not report["missing_channels"]
        )
        report["complete"] = complete
        atomic_json(output.with_suffix(".coverage.json"), report)
        if not complete:
            raise RuntimeError(
                f"nuScenes coverage incomplete: {report['available_images']}/{report['expected_images']} images. "
                f"See {output.with_suffix('.coverage.json')}. Download missing images, then rerun; "
                "completed scenes resume after the remaining sensor archives are extracted."
            )
        temporary = output.with_suffix(".json.tmp")
        counts = Counter()
        with temporary.open("w", encoding="utf-8") as handle:
            handle.write("[")
            first = True
            for shard in shard_paths:
                for row in json.loads(shard.read_text(encoding="utf-8"))["rows"]:
                    if not first:
                        handle.write(",\n")
                    handle.write(json.dumps(row))
                    first = False
                    counts[row["split"]] += 1
            handle.write("]")
        os.replace(temporary, output)
        print(
            f"Prepared {report['available_images']} images; complete={complete}; splits={dict(counts)}",
            flush=True,
        )
        return output

#### Image manifests and data validation

In [ ]:
# fmt: off
#@title Supporting code: Image manifests and data validation
# fmt: on
"""Dataset entry points; nuScenes preparation lives in fog_prepare."""


def resolve_foggy_images(project_root, explicit=None, download=False):
    """Reuse Windows/Linux Kaggle caches from the user's downloader; optional same-source download."""
    import os

    project_root = Path(project_root).resolve()
    candidates = [Path(explicit)] if explicit else []
    candidates += [
        project_root / "data/fog/cityscapes",
        Path.home() / "Downloads" / "foggy-cityscapes",
        Path.home() / "Downloads" / "Foggy Cityscapes",
        project_root
        / ".cache/kagglehub/datasets/minhducnguyen9705/foggy-cityscapes/versions/1",
        Path.home()
        / ".cache/kagglehub/datasets/minhducnguyen9705/foggy-cityscapes/versions/1",
    ]
    for path in candidates:
        if path.is_dir() and (
            (path / "leftImg8bit_foggy").is_dir()
            or (path / "train").is_dir()
            or next(path.rglob("leftImg8bit_foggy"), None) is not None
        ):
            return path
    if not download:
        return None
    # Keep downloads inside this workspace. Version 1 matches Untitled2.ipynb.
    os.environ["KAGGLEHUB_CACHE"] = str(project_root / ".cache/kagglehub")
    try:
        import kagglehub
    except ImportError as error:
        raise ImportError(
            "Install requirements-fog-research.txt to enable the optional Kaggle download"
        ) from error
    print(
        "Downloading the same Foggy Cityscapes v1 source as Untitled2.ipynb (~29.1 GB archive plus extraction).",
        flush=True,
    )
    return Path(
        kagglehub.dataset_download("minhducnguyen9705/foggy-cityscapes/versions/1")
    )


def cityscapes_manifest(
    root, output, with_detection_labels=False, splits=("train", "val")
):
    """Kaggle/official images suffice for density supervision; gtFine is optional.

    Rows without gtFine are explicitly density-only, never negative detection images.
    Set with_detection_labels=True for the optional Foggy Cityscapes detection benchmark.
    """
    root, output = Path(root).resolve(), Path(output)
    if output.exists():
        raise FileExistsError(f"{output} exists. Reuse it or choose a new output path.")
    city_ids = {
        24: "person",
        25: "person",
        26: "car",
        27: "truck",
        28: "bus",
        31: "train",
        32: "motorcycle",
        33: "bicycle",
    }
    rows = []
    from tqdm.auto import tqdm

    # Kaggle extracts may nest leftImg8bit_foggy below additional directories.
    image_roots = (
        [root / "leftImg8bit_foggy"]
        if (root / "leftImg8bit_foggy").is_dir()
        else (
            [root]
            if (root / "train").is_dir()
            else list(root.rglob("leftImg8bit_foggy"))
        )
    )
    if len(image_roots) != 1:
        raise ValueError(
            f"Expected one Foggy Cityscapes image root under {root}; found {len(image_roots)}"
        )
    image_root = image_roots[0]
    for split in splits:
        paths = sorted((image_root / split).glob("*/*_leftImg8bit_foggy_beta_*.png"))
        cache = {}
        for image in tqdm(paths, desc=f"Foggy Cityscapes {split}", mininterval=2):
            base, suffix = image.name.split("_leftImg8bit_foggy_beta_")
            beta = float(suffix.removesuffix(".png"))
            mask_path = (
                root
                / "gtFine"
                / split
                / image.parent.name
                / f"{base}_gtFine_instanceIds.png"
            )
            if base not in cache:
                if with_detection_labels and not mask_path.is_file():
                    raise FileNotFoundError(f"Missing instance labels: {mask_path}")
                boxes, labels, crowds = [], [], []
                with Image.open(image) as source:
                    width, height = source.size
                if with_detection_labels:
                    with Image.open(mask_path) as source:
                        mask = np.array(source)
                    for instance in np.unique(mask):
                        category = int(
                            instance // 1000 if instance >= 1000 else instance
                        )
                        if category in city_ids:
                            yy, xx = np.where(mask == instance)
                            boxes.append(
                                [
                                    int(xx.min()),
                                    int(yy.min()),
                                    int(xx.max()) + 1,
                                    int(yy.max()) + 1,
                                ]
                            )
                            labels.append(CLASSES.index(city_ids[category]))
                            crowds.append(int(instance < 1000))
                cache[base] = boxes, labels, crowds, width, height
            boxes, labels, crowds, width, height = cache[base]
            group = "_".join(base.split("_")[:2])
            bucket = int(hashlib.sha256(group.encode()).hexdigest()[:8], 16) % 10
            if with_detection_labels:
                assigned = (
                    "test"
                    if split == "val"
                    else (
                        "density_train"
                        if bucket < 2
                        else "val"
                        if bucket < 4
                        else "detector_train"
                    )
                )
            else:
                assigned = (
                    "test"
                    if split == "val"
                    else ("val" if bucket < 2 else "density_train")
                )
            rows.append(
                {
                    "id": f"city/{base}/{beta}",
                    "image": str(image),
                    "dataset": "foggy_cityscapes",
                    "group": group,
                    "pair_id": f"city/{base}",
                    "condition": "foggy",
                    "split": assigned,
                    "beta": beta,
                    "boxes": boxes,
                    "labels": labels,
                    "iscrowd": crowds,
                    "width": width,
                    "height": height,
                    "annotation_source": str(mask_path)
                    if with_detection_labels
                    else None,
                    "detection_labeled": with_detection_labels,
                    "label_policy": "instance-mask bounding rectangles; rider merged into person"
                    if with_detection_labels
                    else "beta-only; not a detection target",
                }
            )
    if not rows:
        raise FileNotFoundError(
            f"No official Foggy Cityscapes images under {root}/leftImg8bit_foggy"
        )
    seen = set()
    for row in list(rows):
        base = row["id"].split("/")[1]
        split = "val" if row["split"] == "test" else "train"
        clear = (
            root
            / "leftImg8bit"
            / split
            / base.split("_")[0]
            / f"{base}_leftImg8bit.png"
        )
        if with_detection_labels and not clear.is_file():
            raise FileNotFoundError(f"Missing matched clear image: {clear}")
        if clear.is_file() and base not in seen:
            rows.append(
                {
                    **row,
                    "id": f"city/{base}/clear",
                    "image": str(clear),
                    "beta": 0.0,
                    "condition": "clear",
                }
            )
            seen.add(base)

    atomic_json(output, rows)
    return output

#### Tables, comparisons, and charts

In [ ]:
# fmt: off
#@title Supporting code: Tables, comparisons, and charts
# fmt: on
"""Saved-run analytics: tables, paired changes, per-class AP, density error and plots."""


BALANCED_METRICS = (
    "mAP50_95",
    "AP50",
    "AR100",
    "precision_at_025",
    "recall_at_025",
    "mean_ms",
    "p50_ms",
    "p95_ms",
    "serial_fps",
    "density_mean_ms",
)


def add_balanced_bin_rows(results):
    """Add equal-bin macro summaries without changing natural-distribution rows."""

    if "density_bin" not in results:

        def legacy_bin(value):
            if value in ("all", "unknown", "foggy", "balanced"):
                return value
            return density_bin(float(value))

        results["density_bin"] = results.density.map(legacy_bin)
    if "distribution" not in results:
        results["distribution"] = "natural"
    candidates = results[
        ~results.density.isin(["all", "unknown", "balanced", "foggy"])
        & (results.density_bin != "unknown")
    ]
    summaries = []
    for keys, group in candidates.groupby(["model", "dataset", "arm", "seed"]):
        if group.density_bin.nunique() < 2:
            continue
        summary = group.iloc[0].to_dict()
        summary.update(
            density="balanced",
            density_bin="balanced",
            distribution="balanced_bins",
            images=int(group.images.sum()),
            condition="mixed",
        )
        for column in BALANCED_METRICS:
            if column in group:
                summary[column] = float(
                    group.groupby("density_bin")[column].mean().mean()
                )
        summaries.append(summary)
    if summaries:
        results = pd.concat([results, pd.DataFrame(summaries)], ignore_index=True)
    return results


def export_analytics(output, rows=None):
    output = Path(output)
    if rows is None:
        rows = json.loads((output / INPUT_MANIFEST_FILE).read_text(encoding="utf-8"))
    frames, statuses = [], []
    for directory in sorted(output.iterdir()):
        if not directory.is_dir() or not (directory / STATUS_FILE).exists():
            continue
        status = json.loads((directory / STATUS_FILE).read_text())
        statuses.append(status)
        if status["status"] == "complete" and (directory / RESULTS_FILE).exists():
            frames.append(pd.read_csv(directory / RESULTS_FILE, dtype={"density": str}))
    pd.DataFrame(statuses).to_csv(output / STATUS_TABLE_FILE, index=False)
    health = pd.DataFrame(statuses)
    health_columns = [
        name
        for name in (
            "model",
            "arm",
            "seed",
            "status",
            "planned_epochs",
            "epochs_trained",
            "best_epoch",
            "stop_reason",
        )
        if name in health
    ]
    health[health_columns].to_csv(output / "training-health.csv", index=False)
    if not frames:
        raise RuntimeError(
            f"No completed experiments under {output}. Check status.csv; resume training first."
        )
    results = pd.concat(frames, ignore_index=True)
    settings = {}
    settings_path = output / RUN_SETTINGS_FILE
    if settings_path.exists():
        settings = json.loads(settings_path.read_text(encoding="utf-8"))
    profile = settings.get(
        "profile", settings.get("config", {}).get("profile_name", PROFILE_FULL)
    )
    if isinstance(profile, dict):
        profile = profile.get("name", PROFILE_FULL)
    evaluation_split = settings.get("config", {}).get("evaluation_split", "test")
    if "profile" not in results:
        results["profile"] = profile
    if "evaluation_split" not in results:
        results["evaluation_split"] = evaluation_split
    if "planned_epochs" not in results:
        results["planned_epochs"] = settings.get("config", {}).get("epochs")
    if "epochs_trained" not in results:
        results["epochs_trained"] = np.nan
    if "stop_reason" not in results:
        results["stop_reason"] = "unknown"
    results = add_balanced_bin_rows(results)
    results.to_csv(output / RESULTS_FILE, index=False)
    keys = ["model", "dataset", "seed", "density"]
    control = results[results.arm == ARM_RGB][keys + ["mAP50_95", "mean_ms", "p95_ms"]]
    paired = results[results.arm != ARM_RGB].merge(
        control, on=keys, how="inner", suffixes=("", "_control"), validate="many_to_one"
    )
    paired["delta_mAP"] = paired.mAP50_95 - paired.mAP50_95_control
    paired["delta_latency_ms"] = paired.mean_ms - paired.mean_ms_control
    paired.to_csv(output / PAIRED_DELTA_FILE, index=False)
    export_hypothesis_tables(output, paired)
    metric_names = [
        "mAP50_95",
        "AP50",
        "AR100",
        "precision_at_025",
        "recall_at_025",
        "mean_ms",
        "p95_ms",
        "serial_fps",
    ]
    seed_summary = results.groupby(["dataset", "model", "arm", "density"])[
        metric_names
    ].agg(["mean", "std", "count"])
    seed_summary.to_csv(output / SEED_SUMMARY_FILE)
    by_id = {r["id"]: r for r in rows}
    density_rows = []
    for path in output.glob("density-*/test-predictions.csv"):
        estimates = pd.read_csv(path)
        for dataset, group in estimates.groupby("dataset"):
            error = group.predicted_beta - group.target_beta
            density_rows.append(
                {
                    "experiment": path.parent.name,
                    "dataset": dataset,
                    "images": len(group),
                    "MAE_beta": float(error.abs().mean()),
                    "RMSE_beta": float(np.sqrt((error**2).mean())),
                }
            )
    for path in output.glob("*-predicted_density-*/*-predictions.jsonl"):
        known = []
        with path.open(encoding="utf-8") as handle:
            for line in handle:
                if not line.endswith("\n"):
                    break
                row = json.loads(line)
                target = by_id.get(row["id"], {}).get("beta")
                if target is not None:
                    known.append((row["beta_input"], target))
        if known:
            error = np.array([p - t for p, t in known])
            density_rows.append(
                {
                    "experiment": path.parent.name,
                    "dataset": path.stem.replace("-predictions", ""),
                    "images": len(known),
                    "MAE_beta": float(np.abs(error).mean()),
                    "RMSE_beta": float(np.sqrt((error**2).mean())),
                }
            )
    pd.DataFrame(
        density_rows,
        columns=["experiment", "dataset", "images", "MAE_beta", "RMSE_beta"],
    ).to_csv(output / DENSITY_REPORT_FILE, index=False)
    make_figures(output, results, paired)
    make_hypothesis_figures(output, results, paired, rows)
    return results


def make_figures(output, results, paired):
    import matplotlib

    matplotlib.use(
        "Agg", force=True
    )  # saved figures must work without Tcl/Tk or a desktop
    import matplotlib.pyplot as plt

    figures = output / FIGURE_DIRECTORY
    figures.mkdir(exist_ok=True)
    plt.rcParams.update({"font.size": 10, "figure.dpi": 120})
    styles = {
        ARM_RGB: ("#33658A", "o"),
        ARM_DENSITY: ("#D17A22", "^"),
        ARM_ORACLE: ("#548235", "s"),
    }
    for dataset, frame in results.groupby("dataset"):
        overall = frame[frame.density == "all"]
        if overall.empty:
            continue
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        labels = []
        for position, ((model, arm), part) in enumerate(
            overall.groupby(["model", "arm"])
        ):
            color, marker = styles[arm]
            label = f"{model}\n{arm}"
            labels.append(label)
            values = part.mAP50_95.to_numpy()
            axes[0].bar(position, np.nanmean(values), color=color, alpha=0.65)
            axes[0].scatter(
                np.full(len(values), position),
                values,
                color="black",
                marker=marker,
                s=20,
            )
            axes[1].scatter(
                part.mean_ms,
                part.mAP50_95,
                color=color,
                marker=marker,
                label=f"{model}: {arm}",
            )
        axes[0].set_xticks(range(len(labels)), labels, rotation=25, ha="right")
        axes[0].set(
            ylabel="mAP@[.50:.95]",
            ylim=(0, 1),
            title="Bars: seed mean; points: individual seeds",
        )
        axes[1].set(
            xlabel="End-to-end mean latency (ms/image)",
            ylabel="mAP@[.50:.95]",
            ylim=(0, 1),
            title="Accuracy versus latency",
        )
        axes[1].legend(fontsize=7)
        profiles = ", ".join(sorted(str(value) for value in overall.profile.unique()))
        fig.suptitle(
            f"{dataset} | profile={profiles} | {overall.evaluation_split.iloc[0]} images"
        )
        fig.tight_layout()
        fig.savefig(figures / f"{dataset}-comparison.png", bbox_inches="tight")
        plt.close(fig)
        balanced = frame[frame.density == "balanced"]
        if not balanced.empty:
            fig, ax = plt.subplots(figsize=(10, 5))
            labels = []
            for position, ((model, arm), part) in enumerate(
                balanced.groupby(["model", "arm"])
            ):
                labels.append(f"{model}\n{arm}")
                color, marker = styles[arm]
                ax.bar(position, np.nanmean(part.mAP50_95), color=color, alpha=0.65)
                ax.scatter(
                    np.full(len(part), position),
                    part.mAP50_95,
                    color="black",
                    marker=marker,
                    s=20,
                )
            ax.set_xticks(range(len(labels)), labels, rotation=25, ha="right")
            ax.set(
                ylabel="Equal-beta-bin macro mAP@[.50:.95]",
                ylim=(0, 1),
                title=f"{dataset}: balanced fog-density summary",
            )
            fig.tight_layout()
            fig.savefig(figures / f"{dataset}-balanced.png", bbox_inches="tight")
            plt.close(fig)
        numeric = frame[
            ~frame.density.isin(["all", "unknown", "balanced", "foggy"])
        ].copy()
        if not numeric.empty:
            numeric["beta"] = numeric.density.astype(float)
            fig, ax = plt.subplots(figsize=(9, 5))
            for (model, arm), part in numeric.groupby(["model", "arm"]):
                mean = part.groupby("beta").mAP50_95.mean()
                ax.plot(
                    mean.index,
                    mean.values,
                    marker=styles[arm][1],
                    label=f"{model}: {arm}",
                )
            ax.set(
                xlabel="Known beta (m^-1)",
                ylabel="Mean mAP@[.50:.95]",
                ylim=(0, 1),
                title=f"{dataset}: accuracy by fog level",
            )
            ax.legend(fontsize=8)
            fig.tight_layout()
            fig.savefig(figures / f"{dataset}-density.png")
            plt.close(fig)
        table = overall.groupby(["model", "arm"])[[f"AP_{c}" for c in CLASSES]].mean()
        fig, ax = plt.subplots(figsize=(10, max(3, 0.6 * len(table))))
        plotted = ax.imshow(
            table.to_numpy(), vmin=0, vmax=1, aspect="auto", cmap="viridis"
        )
        ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=30, ha="right")
        ax.set_yticks(range(len(table)), [f"{m}: {a}" for m, a in table.index])
        ax.set_title(f"{dataset}: per-class AP (blank = no ground truth)")
        fig.colorbar(plotted, ax=ax, label="AP@[.50:.95]")
        fig.tight_layout()
        fig.savefig(figures / f"{dataset}-classes.png", bbox_inches="tight")
        plt.close(fig)
        delta = paired[(paired.dataset == dataset) & (paired.density == "all")]
        if not delta.empty:
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            names = []
            for index, ((model, arm), part) in enumerate(
                delta.groupby(["model", "arm"])
            ):
                names.append(f"{model}\n{arm}")
                for ax, column in zip(axes, ["delta_mAP", "delta_latency_ms"]):
                    ax.scatter(
                        np.full(len(part), index), part[column], marker=styles[arm][1]
                    )
            for ax in axes:
                ax.axhline(0, color="black", linewidth=0.8)
                ax.set_xticks(range(len(names)), names, rotation=20, ha="right")
            axes[0].set_ylabel("Paired delta mAP (positive = better)")
            axes[1].set_ylabel("Paired delta latency ms (positive = slower)")
            fig.suptitle(
                f"{dataset}: matched seeds and test images; no significance claim"
            )
            fig.tight_layout()
            fig.savefig(figures / f"{dataset}-deltas.png", bbox_inches="tight")
            plt.close(fig)
    history_files = list(output.glob("*/history.csv"))
    settings_path = output / RUN_SETTINGS_FILE
    if settings_path.exists():
        source = (
            json.loads(settings_path.read_text())
            .get("config", {})
            .get("evaluation_source")
        )
        if source:
            history_files.extend(Path(source).glob("*/history.csv"))
    for history_file in history_files:
        history = pd.read_csv(history_file)
        if not {"train_loss", "val_loss"} <= set(history.columns):
            continue
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(history.epoch, history.train_loss, label="Train", marker="o")
        ax.plot(history.epoch, history.val_loss, label="Tuning", marker="^")
        ax.set(xlabel="Epoch", ylabel="Detector loss", title=history_file.parent.name)
        ax.legend()
        fig.tight_layout()
        fig.savefig(figures / f"{history_file.parent.name}-loss.png")
        plt.close(fig)


def export_hypothesis_tables(output, paired):
    """Compare density versus control separately on clear and pooled fog images."""
    keys = ["dataset", "model", "arm", "seed"]
    clear = paired[paired.density == "0.0"][keys + ["delta_mAP", "images"]]
    fog = paired[paired.density == "foggy"][keys + ["delta_mAP", "images"]]
    effects = clear.merge(
        fog, on=keys, suffixes=("_clear", "_fog"), validate="one_to_one"
    )
    effects["fog_advantage"] = effects.delta_mAP_fog - effects.delta_mAP_clear
    effects["supports_direction_in_this_seed"] = (effects.delta_mAP_fog > 0) & (
        effects.delta_mAP_clear < 0
    )
    effects.to_csv(output / HYPOTHESIS_FILE, index=False)
    summaries = []
    for keys, group in effects.groupby(["dataset", "model", "arm"]):
        summary = dict(zip(["dataset", "model", "arm"], keys))
        summary["seeds"] = group.seed.nunique()
        summary["interpretation"] = "Exploratory; no seed uncertainty with one seed"
        for metric in ("delta_mAP_clear", "delta_mAP_fog", "fog_advantage"):
            values = group[metric].to_numpy()
            summary[metric + "_mean"] = float(values.mean())
            summary[metric + "_std"] = (
                float(values.std(ddof=1)) if len(values) > 1 else np.nan
            )
            lower = upper = np.nan
            if len(values) >= 3:
                samples = np.random.default_rng(42).choice(values, (5000, len(values)))
                lower, upper = np.percentile(samples.mean(axis=1), [2.5, 97.5])
                summary["interpretation"] = (
                    "95% bootstrap interval over training seeds; few seeds remain exploratory"
                )
            summary[metric + "_ci_low"] = lower
            summary[metric + "_ci_high"] = upper
        summaries.append(summary)
    pd.DataFrame(summaries).to_csv(output / "hypothesis-summary.csv", index=False)


def make_hypothesis_figures(output, results, paired, rows):
    import matplotlib.pyplot as plt

    figures = output / FIGURE_DIRECTORY
    for dataset, frame in results.groupby("dataset"):
        conditions = frame[frame.density.isin(["0.0", "foggy"])]
        if conditions.empty:
            continue
        fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
        for ax, (value, title) in zip(axes, [("0.0", "Clear"), ("foggy", "Foggy")]):
            part = conditions[conditions.density == value]
            table = part.groupby(["model", "arm"]).mAP50_95.mean().unstack()
            arm_order = [
                arm for arm in (ARM_RGB, ARM_DENSITY, ARM_ORACLE) if arm in table
            ]
            table[arm_order].rename(
                columns={
                    ARM_RGB: "RGB control",
                    ARM_DENSITY: "Predicted density",
                    ARM_ORACLE: "Known density (oracle)",
                }
            ).plot.bar(ax=ax, color=["#33658A", "#D17A22", "#548235"][: len(arm_order)])
            count = part.images.max()
            ax.set(
                title=f"{title}: {count} images per model/arm",
                xlabel="Model",
                ylabel="Detection mAP (higher is better)",
                ylim=(0, 1),
            )
            ax.tick_params(axis="x", rotation=0)
            ax.legend(fontsize=7)
        fig.suptitle(f"{dataset}: same scenes and labels; bars average training seeds")
        fig.tight_layout()
        fig.savefig(figures / f"{dataset}-conditions.png", bbox_inches="tight")
        plt.close(fig)
        delta = paired[
            (paired.dataset == dataset) & paired.density.isin(["0.0", "foggy"])
        ]
        if not delta.empty:
            fig, ax = plt.subplots(figsize=(12, 5))
            table = (
                delta.groupby(["model", "arm", "density"]).delta_mAP.mean().unstack()
            )
            table.rename(columns={"0.0": "Clear", "foggy": "Foggy"}).plot.bar(ax=ax)
            ax.set_xticklabels(
                [f"{model}\n{arm.replace('_', ' ')}" for model, arm in table.index],
                rotation=0,
            )
            ax.axhline(0, color="black", linewidth=1)
            ax.set(
                ylabel="Density minus RGB control: delta mAP",
                xlabel="Model and density arm",
                title="Above zero helps; below zero hurts. One seed cannot establish significance.",
            )
            fig.tight_layout()
            fig.savefig(figures / f"{dataset}-effects.png", bbox_inches="tight")
            plt.close(fig)
        columns = [f"AP_{name}" for name in CLASSES]
        table = conditions.groupby(["density", "model", "arm"])[columns].mean()
        fig, ax = plt.subplots(figsize=(11, max(4, len(table) * 0.4)))
        heat = ax.imshow(table.to_numpy(), vmin=0, vmax=1, aspect="auto")
        ax.set_xticks(range(len(columns)), CLASSES, rotation=30)
        ax.set_yticks(range(len(table)), [" | ".join(key) for key in table.index])
        ax.set_title("Per-class AP by condition (blank = no labeled examples)")
        fig.colorbar(heat, ax=ax, label="AP: higher is better")
        fig.tight_layout()
        fig.savefig(figures / f"{dataset}-condition-classes.png", bbox_inches="tight")
        plt.close(fig)
    audit = pd.DataFrame(
        [
            {
                "dataset": row["dataset"],
                "split": row["split"],
                "bin": density_bin(row["beta"]),
                "detection_labeled": row.get("detection_labeled", True),
                "pair_id": row.get("pair_id"),
                "objects": len(row["labels"]),
            }
            for row in rows
        ]
    )
    if not audit.empty:
        audit.groupby(["dataset", "split", "bin", "detection_labeled"]).agg(
            images=("dataset", "size"),
            pairs=("pair_id", "nunique"),
            objects=("objects", "sum"),
        ).to_csv(output / "dataset-audit.csv")
        classes = [
            {
                "dataset": row["dataset"],
                "split": row["split"],
                "bin": density_bin(row["beta"]),
                "class": CLASSES[label],
                "crowd": crowd,
            }
            for row in rows
            for label, crowd in zip(
                row["labels"], row.get("iscrowd", [0] * len(row["labels"]))
            )
        ]
        if classes:
            pd.DataFrame(classes).groupby(
                ["dataset", "split", "bin", "class", "crowd"]
            ).size().rename("objects").to_csv(output / "class-coverage.csv")
        fig, ax = plt.subplots(figsize=(12, 5))
        audit.groupby(["dataset", "split", "bin"]).size().plot.bar(ax=ax)
        ax.set(
            ylabel="Images",
            xlabel="Dataset / split / fog level",
            title="Data balance: unknown fog condition is not clear",
        )
        fig.tight_layout()
        fig.savefig(figures / "dataset-audit.png", bbox_inches="tight")
        plt.close(fig)
    make_density_diagnostics(output, rows)
    make_precision_recall_figures(output)


def make_density_diagnostics(output, rows):
    import matplotlib.pyplot as plt

    lookup = {row["id"]: row for row in rows}
    estimates = []
    for path in output.glob("*-predicted_density-*/*-predictions.jsonl"):
        for line in path.read_text().splitlines(keepends=True):
            if not line.endswith("\n"):
                break
            record = json.loads(line)
            target = lookup.get(record["id"], {}).get("beta")
            if target is not None:
                estimates.append(
                    {
                        "experiment": path.parent.name,
                        "id": record["id"],
                        "target": target,
                        "predicted": record["beta_input"],
                    }
                )
    if not estimates:
        return
    data = pd.DataFrame(estimates)
    data["error"] = data.predicted - data.target
    data["bin"] = data.target.map(density_bin)
    data.to_csv(output / "density-estimates.csv", index=False)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].scatter(data.target, data.predicted, s=8, alpha=0.3)
    axes[0].plot([0, 0.02], [0, 0.02], color="black")
    axes[0].set(
        xlabel="Known beta (m^-1)",
        ylabel="Predicted beta (m^-1)",
        title="Closer to diagonal = better density estimate",
    )
    data.boxplot(column="error", by="bin", ax=axes[1])
    axes[1].axhline(0, color="black")
    axes[0].set_xticks([0, 0.005, 0.01, 0.02])
    axes[1].set(title="Density error by fog level", ylabel="Predicted minus known beta")
    axes[2].hist(data.error, bins=25)
    axes[2].set(title="Density errors", xlabel="Beta error", ylabel="Predictions")
    axes[2].set_xticks([-0.01, -0.005, 0, 0.005, 0.01])
    fig.suptitle("Density quality is distinct from bounding-box detection quality")
    fig.tight_layout()
    fig.savefig(output / "figures/density-diagnostics.png", bbox_inches="tight")
    plt.close(fig)


def make_precision_recall_figures(output):
    import matplotlib.pyplot as plt

    records = []
    for path in output.glob("*/*-precision-recall.json"):
        records.extend(json.loads(path.read_text()))
    if not records:
        return
    curves = pd.DataFrame(records)
    curves.to_json(output / "precision-recall.json", orient="records", indent=2)
    for dataset, frame in curves.groupby("dataset"):
        fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)
        for ax, condition in zip(axes, ["clear", "foggy"]):
            for (model, arm), group in frame[frame.condition == condition].groupby(
                ["model", "arm"]
            ):
                precision = np.array(group.precision.tolist(), dtype=float)
                mean = np.nanmean(precision, axis=0)
                ax.plot(group.iloc[0].recall, mean, label=f"{model}: {arm}")
            ax.set(
                xlabel="Recall: fraction of labeled objects found",
                ylabel="Precision: fraction of reported objects correct",
                title=f"{condition.title()} images",
                xlim=(0, 1),
                ylim=(0, 1.02),
            )
            if ax.lines:
                ax.legend(fontsize=7)
        fig.suptitle(
            f"{dataset}: precision versus recall at IoU 0.50; class macro average"
        )
        fig.tight_layout()
        fig.savefig(
            output / FIGURE_DIRECTORY / f"{dataset}-precision-recall.png",
            bbox_inches="tight",
        )
        plt.close(fig)

#### Detector training and checkpoint recovery

In [ ]:
# fmt: off
#@title Supporting code: Detector training and checkpoint recovery
# fmt: on
"""Full detector fine-tuning, resumable checkpoints, preflight, and persistent evaluation."""


def default_config(root):
    """Full-run settings for local command-line use, from the same notebook defaults."""
    root = Path(root).resolve()
    full = PROFILE_SETTINGS[PROFILE_FULL]
    return {
        **deepcopy(TRAINING_DEFAULTS),
        "output": str(root / "reports/fog/full-training-complete"),
        **{key: str(root / value) for key, value in MANIFEST_PATHS.items()},
        "checkpoints": {
            name: checkpoint if name == DETR_MODEL else str(root / checkpoint)
            for name, checkpoint in MODEL_SPECS.items()
        },
        "arms": list(DETECTION_ARMS),
        "seeds": list(FULL_RUN_SEEDS),
        "device": "cuda:0" if torch.cuda.is_available() else "cpu",
        "amp": True,
        "size": full["size"],
        "epochs": full["epochs"],
        "save_every": full["save_every"],
        "profile_name": PROFILE_FULL,
        "initialization": "existing_full_run",
        "evaluation_split": full["evaluation_split"],
        "early_stopping_patience": full["early_stopping_patience"],
    }


def summarize_cumulative_training_progress(progress, epochs, images_per_epoch):
    total_image_steps = epochs * images_per_epoch
    completed_image_steps = min(
        total_image_steps, progress.epoch * images_per_epoch + progress.step
    )
    return {
        "completed_epochs": progress.epoch,
        "planned_epochs": epochs,
        "saved_image_step_in_epoch": progress.step,
        "images_per_epoch": images_per_epoch,
        "completed_image_steps": completed_image_steps,
        "planned_image_steps": total_image_steps,
        "overall_training_percent": 100 * completed_image_steps / total_image_steps
        if total_image_steps
        else 0.0,
    }


def display_checkpoint_continuation(session, epochs, annotations, balanced=False):
    summary = summarize_cumulative_training_progress(
        session.progress, epochs, epoch_size(annotations, balanced)
    )
    print(f"Checkpoint continuation: {session.checkpoint_directory}")
    print(
        f"Overall training: {summary['overall_training_percent']:.2f}% | "
        f"completed epochs {summary['completed_epochs']}/{epochs} | "
        f"saved image steps {summary['saved_image_step_in_epoch']}/{len(annotations)}"
    )
    if session.progress.history:
        print("Cumulative loss history, including previous sessions:")
        print(pd.DataFrame(session.progress.history).to_string(index=False))


def read_run_continuation_summary(configuration):
    output_directory = Path(configuration["output"])
    expected_runs = []
    for seed in configuration["seeds"]:
        if ARM_DENSITY in configuration["arms"] and not configuration.get(
            "density_checkpoint"
        ):
            expected_runs.append(
                (f"{DENSITY_RUN_PREFIX}-{seed}", configuration["density_epochs"])
            )
        for model in configuration["checkpoints"]:
            for arm in configuration["arms"]:
                expected_runs.append((f"{model}-{arm}-{seed}", configuration["epochs"]))
    records = []
    for run_name, epochs in expected_runs:
        directory = output_directory / run_name
        checkpoint_path = directory / LAST_CHECKPOINT
        record = {
            "run": run_name,
            "completed_epochs": 0,
            "planned_epochs": epochs,
            "saved_image_step_in_epoch": 0,
            "overall_training_percent": 0.0,
            "status": "not started",
        }
        if checkpoint_path.exists():
            saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
            progress = DetectorTrainingProgress(
                **{
                    key: saved[key]
                    for key in (
                        "epoch",
                        "step",
                        "best_loss",
                        "running_sum",
                        "running_count",
                        "history",
                    )
                }
            )
            manifest_path = output_directory / INPUT_MANIFEST_FILE
            if manifest_path.is_file():
                manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
                split = (
                    "density_train"
                    if run_name.startswith(f"{DENSITY_RUN_PREFIX}-")
                    else "detector_train"
                )
                record.update(
                    summarize_cumulative_training_progress(
                        progress,
                        epochs,
                        epoch_size(
                            partitions(manifest)[split],
                            split == "density_train"
                            and configuration.get("density_sampling", "natural")
                            == "balanced_bins",
                        ),
                    )
                )
                record["status"] = (
                    "training complete"
                    if progress.epoch >= epochs
                    else "checkpoint saved"
                )
            else:
                # A prior process can leave last.pt before run identity is recorded.
                # Keep the progress table usable and let run_training issue its
                # safer "choose a new output folder" error for that orphan.
                record["status"] = "orphan checkpoint (manifest missing)"
        if (directory / STATUS_FILE).exists():
            record["status"] = json.loads((directory / STATUS_FILE).read_text())[
                "status"
            ]
        records.append(record)
    return pd.DataFrame(records)


def fingerprint(rows):
    # Annotation/identity fingerprint plus image stat records; does not re-read every JPEG.
    value = hashlib.sha256()
    for row in sorted(rows, key=lambda r: r["id"]):
        stat = Path(row["image"]).stat()
        value.update(
            json.dumps([row, stat.st_size, stat.st_mtime_ns], sort_keys=True).encode()
        )
    return value.hexdigest()


def partitions(rows, config=None):
    parts = {
        name: [
            r
            for r in rows
            if r["split"] == name
            and (name == "density_train" or r.get("detection_labeled", True))
            and (name not in ("detector_train", "val") or not any(r.get("iscrowd", [])))
        ]
        for name in ("detector_train", "density_train", "val", "test")
    }
    parts["density_val"] = [
        r for r in rows if r["split"] == "val" and r["beta"] is not None
    ]
    parts["density_test"] = [
        r for r in rows if r["split"] == "test" and r["beta"] is not None
    ]
    if config and config.get("evaluation_split") == "val":
        parts["evaluation"] = parts["val"]
        parts["density_evaluation"] = parts["density_val"]
    else:
        parts["evaluation"] = parts["test"]
        parts["density_evaluation"] = parts["density_test"]
    return parts


def preflight(config, rows):
    """Return actionable checks without starting training or downloading anything."""
    parts = partitions(rows, config)
    checks = []

    def check(name, good, detail):
        checks.append({"check": name, "ready": bool(good), "detail": detail})

    check(
        "training images",
        bool(parts["detector_train"]),
        f"{len(parts['detector_train'])} labeled images",
    )
    check(
        "tuning images",
        bool(parts["val"]),
        f"{len(parts['val'])} scene-held-out images for checkpoint selection",
    )
    evaluation_split = config.get("evaluation_split", "test")
    check(
        "evaluation split",
        evaluation_split in VALID_EVALUATION_SPLITS,
        evaluation_split,
    )
    check(
        "evaluation images",
        bool(parts["evaluation"]),
        f"{len(parts['evaluation'])} {evaluation_split} images for metrics",
    )
    valid_arms = {ARM_RGB, ARM_DENSITY, ARM_ORACLE}
    check(
        "arms",
        bool(config["arms"])
        and len(set(config["arms"])) == len(config["arms"])
        and set(config["arms"]) <= valid_arms,
        ", ".join(config["arms"]),
    )
    groups = {}
    clean = True
    ids = set()
    for row in rows:
        key = (row["dataset"], row["group"])
        clean &= key not in groups or groups[key] == row["split"]
        clean &= row["id"] not in ids
        groups[key] = row["split"]
        ids.add(row["id"])
    check(
        "scene/ID isolation",
        clean,
        "No scene crosses train, tuning, density-training or final-test splits",
    )
    for key in (
        "size",
        "epochs",
        "density_epochs",
        "accumulate",
        "save_every",
        "repeats",
    ):
        check(key, isinstance(config[key], int) and config[key] > 0, str(config[key]))
    check(
        "size alignment", config["size"] % 32 == 0, "Image size must be divisible by 32"
    )
    check(
        "seeds",
        bool(config["seeds"]) and len(set(config["seeds"])) == len(config["seeds"]),
        str(config["seeds"]),
    )
    profile = config.get("profile_name", PROFILE_FULL)
    check("profile", profile in PROFILE_NAMES, profile)
    check(
        "initialization",
        config.get("initialization", "pretrained")
        in {"pretrained", "existing_full_run"},
        config.get("initialization", "pretrained"),
    )
    check(
        "batch contract",
        config.get("batch_size", 1) == 1
        and config.get("effective_batch_size", config.get("accumulate", 1))
        == config.get("accumulate", 1),
        "real detector training uses batch_size=1 with gradient accumulation",
    )
    density_sampling = config.get("density_sampling", "natural")
    check(
        "density sampling",
        density_sampling in {"natural", "balanced_bins"},
        density_sampling,
    )
    coverage_path = Path(config["manifest"]).with_suffix(".coverage.json")
    if config.get("test_scope", {}).get("enabled"):
        check(
            "subset test scope",
            bool(rows),
            "Small-data execution test; not full-study coverage or scientific results",
        )
    elif any(r["dataset"] == "nuscenes" for r in rows):
        coverage = (
            json.loads(coverage_path.read_text()) if coverage_path.exists() else {}
        )
        complete = coverage.get("complete") is True
        check(
            "nuScenes coverage",
            complete,
            f"complete={complete}; coverage report: {coverage_path}. All expected camera keyframes are required.",
        )
    if ARM_DENSITY in config["arms"]:
        source = config.get("density_checkpoint")
        if source:
            try:
                saved = torch.load(source, map_location="cpu", weights_only=True)
                correct = (
                    saved.get("kind") == "fog-density-v1"
                    and saved.get("beta_scale") == BETA_SCALE
                )
                trained_groups = {tuple(x) for x in saved.get("train_groups", [])}
                held_out = {
                    (r["dataset"], r["group"])
                    for r in rows
                    if r["split"] in ("val", "test", "detector_train")
                }
                correct &= bool(trained_groups) and not (trained_groups & held_out)
                check(
                    "density checkpoint",
                    correct,
                    "Requires this pipeline format, beta units, and independent training groups",
                )
            except Exception as error:
                check("density checkpoint", False, f"{type(error).__name__}: {error}")
        else:
            train = parts["density_train"]
            known_val = parts["density_val"]
            check(
                "density supervision",
                bool(train)
                and all(r["beta"] is not None for r in train)
                and len({r["beta"] for r in train}) > 1
                and bool(known_val),
                "Add Foggy Cityscapes images (beta comes from filenames; gtFine is not required), prepare its manifest, "
                "or set density_checkpoint to an independently trained fog-density checkpoint. "
                "Native nuScenes has no beta labels; its visibility field is NOT fog density.",
            )
    if ARM_ORACLE in config["arms"]:
        check(
            "oracle labels",
            all(
                r["beta"] is not None
                for r in parts["detector_train"] + parts["val"] + parts["test"]
            ),
            "Oracle is optional and requires known beta for every matched train/validation/test image. "
            "Use the default rgb_control/predicted_density comparison for native nuScenes.",
        )
    check(
        "models",
        bool(config["checkpoints"])
        and set(config["checkpoints"]) <= {YOLO_SMALL, YOLO_MEDIUM, DETR_MODEL},
        "Three supported pretrained architectures; missing public weights download on first load",
    )
    check(
        "device",
        not config["device"].startswith("cuda") or torch.cuda.is_available(),
        config["device"],
    )
    return pd.DataFrame(checks)


def save_torch(path, payload):
    temporary = path.with_suffix(".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


@dataclass
class DensityTrainingSession:
    model: object
    optimizer: object
    dataset_partitions: dict
    configuration: dict
    checkpoint_directory: Path
    seed: int
    progress: object


def restore_density_training_checkpoint(session):
    checkpoint_path = session.checkpoint_directory / LAST_CHECKPOINT
    if not checkpoint_path.exists():
        return
    saved_checkpoint = torch.load(
        checkpoint_path, map_location=session.configuration["device"], weights_only=True
    )
    session.model.load_state_dict(saved_checkpoint["model"])
    session.optimizer.load_state_dict(saved_checkpoint["optimizer"])
    session.progress = DetectorTrainingProgress(
        **{
            name: saved_checkpoint[name]
            for name in (
                "epoch",
                "step",
                "best_loss",
                "running_sum",
                "running_count",
                "history",
            )
        }
    )

    display_checkpoint_continuation(
        session,
        session.configuration["density_epochs"],
        session.dataset_partitions["density_train"],
        balanced=session.configuration.get("density_sampling") == "balanced_bins",
    )


def save_density_training_checkpoint(session, improved=False):
    training_groups = sorted(
        {
            (annotation["dataset"], annotation["group"])
            for annotation in session.dataset_partitions["density_train"]
        }
    )
    payload = {
        "kind": "fog-density-v1",
        "beta_scale": BETA_SCALE,
        "model": session.model.state_dict(),
        "optimizer": session.optimizer.state_dict(),
        **vars(session.progress),
        "train_groups": training_groups,
    }
    if improved:
        save_torch(session.checkpoint_directory / BEST_CHECKPOINT, payload)
    save_torch(session.checkpoint_directory / LAST_CHECKPOINT, payload)


def train_one_density_epoch(session):
    configuration = session.configuration
    progress = session.progress
    training_annotations = session.dataset_partitions["density_train"]
    image_order = epoch_indices(
        training_annotations,
        session.seed,
        progress.epoch,
        balanced=configuration.get("density_sampling", "natural") == "balanced_bins",
    )
    session.model.train()
    description = f"Density seed={session.seed} epoch={progress.epoch + 1}"
    for image_step in tqdm(
        range(progress.step, len(image_order)),
        total=len(image_order),
        initial=progress.step,
        desc=description,
        mininterval=2,
    ):
        annotation = training_annotations[image_order[image_step]]
        image = image_tensor(annotation, configuration["size"], configuration["device"])
        target_density = image.new_tensor([[annotation["beta"]]])
        session.optimizer.zero_grad(set_to_none=True)
        loss = torch.nn.functional.mse_loss(
            session.model(image) / BETA_SCALE, target_density / BETA_SCALE
        )
        if not torch.isfinite(loss):
            raise ValueError("Nonfinite density loss")
        loss.backward()
        session.optimizer.step()
        progress.running_sum += float(loss.detach())
        progress.running_count += 1
        progress.step = image_step + 1
        if progress.step % configuration["save_every"] == 0:
            save_density_training_checkpoint(session)


def measure_density_validation_loss(session):
    session.model.eval()
    squared_errors = []
    with torch.inference_mode():
        for annotation in tqdm(
            session.dataset_partitions["density_val"],
            desc="Density tuning",
            mininterval=2,
        ):
            image = image_tensor(
                annotation,
                session.configuration["size"],
                session.configuration["device"],
            )
            predicted_density = session.model(image).item()
            squared_errors.append((predicted_density - annotation["beta"]) ** 2)
    return float(np.mean(squared_errors))


def record_completed_density_epoch(session, validation_loss):
    progress = session.progress
    progress.history.append(
        {
            "epoch": progress.epoch + 1,
            "train_normalized_mse": progress.running_sum / progress.running_count,
            "val_beta_rmse": validation_loss**0.5,
        }
    )
    improved = validation_loss < progress.best_loss
    progress.best_loss = min(progress.best_loss, validation_loss)
    progress.epoch += 1
    progress.step = 0
    progress.running_sum, progress.running_count = 0.0, 0
    save_density_training_checkpoint(session, improved)
    pd.DataFrame(progress.history).to_csv(
        session.checkpoint_directory / LOSS_HISTORY_FILE, index=False
    )


def should_stop_early(history, metric, patience):
    if not isinstance(patience, int) or patience <= 0 or len(history) <= patience:
        return False
    prior_best = min(row[metric] for row in history[:-patience])
    return all(row[metric] >= prior_best for row in history[-patience:])


def fit_density(parts, config, directory, seed):
    """Restore the density estimator, train each epoch and select its best weights."""
    directory.mkdir(parents=True, exist_ok=True)
    seed_all(seed)
    model = DensityCNN().to(config["device"])
    source_checkpoint = config.get("density_checkpoint")
    if source_checkpoint:
        saved_checkpoint = torch.load(
            source_checkpoint, map_location=config["device"], weights_only=True
        )
        model.load_state_dict(saved_checkpoint["model"])
        return model.eval().requires_grad_(False)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    session = DensityTrainingSession(
        model, optimizer, parts, config, directory, seed, DetectorTrainingProgress()
    )
    restore_density_training_checkpoint(session)
    while session.progress.epoch < config["density_epochs"]:
        train_one_density_epoch(session)
        validation_loss = measure_density_validation_loss(session)
        record_completed_density_epoch(session, validation_loss)
        if should_stop_early(
            session.progress.history,
            "val_beta_rmse",
            config.get("early_stopping_patience", 0),
        ):
            progress_path = directory / PROGRESS_FILE
            atomic_json(
                progress_path,
                {"epoch": session.progress.epoch, "reason": "early_stopping"},
            )
            break
    best_checkpoint = torch.load(
        directory / BEST_CHECKPOINT, map_location=config["device"], weights_only=True
    )
    model.load_state_dict(best_checkpoint["model"])
    return model.eval().requires_grad_(False)


@dataclass
class DetectorTrainingProgress:
    """Fields persisted in existing detector checkpoints; their names are stable."""

    epoch: int = 0
    step: int = 0
    best_loss: float = float("inf")
    running_sum: float = 0.0
    running_count: int = 0
    history: list = field(default_factory=list)


@dataclass
class DetectorTrainingSession:
    detector: object
    adapter: object
    density_estimator: object
    dataset_partitions: dict
    comparison_arm: str
    configuration: dict
    checkpoint_directory: Path
    seed: int
    optimizer: object
    gradient_scaler: object
    detector_parameters: list
    mixed_precision_enabled: bool
    progress: DetectorTrainingProgress = field(default_factory=DetectorTrainingProgress)


def create_detector_training_session(
    detector, adapter, density, parts, arm, config, directory, seed
):
    detector_parameters = [
        parameter
        for parameter in detector.model.parameters()
        if parameter.requires_grad
    ]
    optimizer = torch.optim.AdamW(
        [
            {
                "params": detector_parameters,
                "lr": config["learning_rates"][detector.name],
            },
            {"params": adapter.parameters(), "lr": config["adapter_lr"]},
        ],
        weight_decay=1e-4,
    )
    mixed_precision_enabled = config["amp"] and config["device"].startswith("cuda")
    gradient_scaler = torch.amp.GradScaler(
        "cuda", enabled=mixed_precision_enabled, init_scale=128.0
    )
    return DetectorTrainingSession(
        detector,
        adapter,
        density,
        parts,
        arm,
        config,
        directory,
        seed,
        optimizer,
        gradient_scaler,
        detector_parameters,
        mixed_precision_enabled,
    )


def restore_detector_training_checkpoint(session):
    checkpoint_path = session.checkpoint_directory / LAST_CHECKPOINT
    if not checkpoint_path.exists():
        return
    saved_checkpoint = torch.load(
        checkpoint_path, map_location=session.configuration["device"], weights_only=True
    )
    session.detector.model.load_state_dict(saved_checkpoint["model"])
    session.adapter.load_state_dict(saved_checkpoint["adapter"])
    session.optimizer.load_state_dict(saved_checkpoint["optimizer"])
    session.gradient_scaler.load_state_dict(saved_checkpoint["scaler"])
    session.progress = DetectorTrainingProgress(
        **{
            name: saved_checkpoint[name]
            for name in (
                "epoch",
                "step",
                "best_loss",
                "running_sum",
                "running_count",
                "history",
            )
        }
    )
    torch.set_rng_state(saved_checkpoint["rng"].cpu())
    if session.configuration["device"].startswith("cuda"):
        torch.cuda.set_rng_state_all(
            [state.cpu() for state in saved_checkpoint["cuda_rng"]]
        )

    display_checkpoint_continuation(
        session,
        session.configuration["epochs"],
        session.dataset_partitions["detector_train"],
    )


def save_detector_training_checkpoint(session, improved=False, reason="periodic"):
    progress = session.progress
    payload = {
        "kind": "fog-detector-v1",
        "model": session.detector.model.state_dict(),
        "adapter": session.adapter.state_dict(),
        "optimizer": session.optimizer.state_dict(),
        "scaler": session.gradient_scaler.state_dict(),
        **vars(progress),
        "rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all()
        if session.configuration["device"].startswith("cuda")
        else [],
        "checkpoint_reason": reason,
    }
    if improved:
        save_torch(session.checkpoint_directory / BEST_CHECKPOINT, payload)
    save_torch(session.checkpoint_directory / LAST_CHECKPOINT, payload)
    atomic_json(
        session.checkpoint_directory / PROGRESS_FILE,
        {
            "model": session.detector.name,
            "arm": session.comparison_arm,
            "seed": session.seed,
            "epoch": progress.epoch,
            "step": progress.step,
            "best_loss": float(progress.best_loss)
            if np.isfinite(progress.best_loss)
            else None,
            "reason": reason,
            **summarize_cumulative_training_progress(
                progress,
                session.configuration["epochs"],
                len(session.dataset_partitions["detector_train"]),
            ),
        },
    )


def calculate_finite_detector_loss(session, image, annotation, fog_density):
    with torch.autocast(
        device_type=image.device.type, enabled=session.mixed_precision_enabled
    ):
        loss = session.detector.loss(session.adapter(image, fog_density), annotation)
    if torch.isfinite(loss):
        return loss
    if session.mixed_precision_enabled:
        # Some examples overflow in float16; retry the same loss in float32.
        with torch.autocast(device_type=image.device.type, enabled=False):
            loss = session.detector.loss(
                session.adapter(image.float(), fog_density.float()), annotation
            )
        if torch.isfinite(loss):
            return loss
    raise ValueError(
        f"Nonfinite loss for {annotation['id']}; AMP fallback was also nonfinite"
    )


def apply_accumulated_detector_gradients(session):
    session.gradient_scaler.unscale_(session.optimizer)
    torch.nn.utils.clip_grad_norm_(
        session.detector_parameters + list(session.adapter.parameters()), 10
    )
    session.gradient_scaler.step(session.optimizer)
    session.gradient_scaler.update()
    session.optimizer.zero_grad(set_to_none=True)


def capture_detector_random_state(configuration):
    cpu_state = torch.get_rng_state()
    cuda_states = (
        torch.cuda.get_rng_state_all()
        if configuration["device"].startswith("cuda")
        else []
    )
    return cpu_state, cuda_states


def restore_detector_random_state(configuration, random_state):
    cpu_state, cuda_states = random_state
    torch.set_rng_state(cpu_state)
    if configuration["device"].startswith("cuda"):
        torch.cuda.set_rng_state_all(cuda_states)


def accumulate_one_detector_image(session, annotation, accumulation_window_size):
    configuration = session.configuration
    image = image_tensor(annotation, configuration["size"], configuration["device"])
    fog_density = beta_for(
        session.comparison_arm, image, annotation, session.density_estimator
    )
    loss = calculate_finite_detector_loss(session, image, annotation, fog_density)
    session.gradient_scaler.scale(loss / accumulation_window_size).backward()
    session.progress.running_sum += float(loss.detach())
    session.progress.running_count += 1


def checkpoint_completed_optimizer_update(session, next_image_step, image_count):
    apply_accumulated_detector_gradients(session)
    session.progress.step = next_image_step
    completed_window_count = next_image_step // session.configuration["accumulate"]
    checkpoint_due = completed_window_count % session.configuration["save_every"] == 0
    if checkpoint_due or next_image_step == image_count:
        save_detector_training_checkpoint(session)


def train_detector_microsteps(session, image_order):
    configuration = session.configuration
    progress = session.progress
    completed_loss_statistics = progress.running_sum, progress.running_count
    completed_random_state = capture_detector_random_state(configuration)
    description = f"{session.detector.name}/{session.comparison_arm} seed={session.seed} epoch={progress.epoch + 1}"
    image_steps = tqdm(
        range(progress.step, len(image_order)),
        total=len(image_order),
        initial=progress.step,
        desc=description,
        mininterval=2,
    )
    try:
        for image_step in image_steps:
            annotation = session.dataset_partitions["detector_train"][
                image_order[image_step]
            ]
            window_start = (image_step // configuration["accumulate"]) * configuration[
                "accumulate"
            ]
            window_size = min(
                configuration["accumulate"], len(image_order) - window_start
            )
            accumulate_one_detector_image(session, annotation, window_size)
            next_image_step = image_step + 1
            optimizer_boundary = next_image_step % configuration[
                "accumulate"
            ] == 0 or next_image_step == len(image_order)
            if optimizer_boundary:
                checkpoint_completed_optimizer_update(
                    session, next_image_step, len(image_order)
                )
                completed_loss_statistics = progress.running_sum, progress.running_count
                completed_random_state = capture_detector_random_state(configuration)
    except KeyboardInterrupt:
        # Replay the partial window with its original RNG state and loss statistics.
        session.optimizer.zero_grad(set_to_none=True)
        progress.running_sum, progress.running_count = completed_loss_statistics
        restore_detector_random_state(configuration, completed_random_state)
        save_detector_training_checkpoint(session, reason="interrupt")
        raise


def train_one_detector_epoch(session):
    session.detector.training_mode()
    session.adapter.train()
    training_image_count = len(session.dataset_partitions["detector_train"])
    image_order = random.Random(session.seed + session.progress.epoch).sample(
        range(training_image_count), training_image_count
    )
    session.optimizer.zero_grad(set_to_none=True)
    train_detector_microsteps(session, image_order)


def measure_detector_validation_loss(session):
    session.detector.model.eval()
    session.adapter.eval()
    validation_losses = []
    description = f"{session.detector.name}/{session.comparison_arm} tuning"
    with torch.no_grad():
        for annotation in tqdm(
            session.dataset_partitions["val"], desc=description, mininterval=2
        ):
            image = image_tensor(
                annotation,
                session.configuration["size"],
                session.configuration["device"],
            )
            fog_density = beta_for(
                session.comparison_arm, image, annotation, session.density_estimator
            )
            loss = calculate_finite_detector_loss(
                session, image, annotation, fog_density
            )
            validation_losses.append(float(loss))
    validation_loss = float(np.mean(validation_losses))
    if not np.isfinite(validation_loss):
        raise ValueError("Nonfinite validation loss")
    return validation_loss


def record_completed_detector_epoch(session, validation_loss):
    progress = session.progress
    progress.history.append(
        {
            "epoch": progress.epoch + 1,
            "train_loss": progress.running_sum / progress.running_count,
            "val_loss": validation_loss,
        }
    )
    improved = validation_loss < progress.best_loss
    progress.best_loss = min(progress.best_loss, validation_loss)
    progress.epoch += 1
    progress.step = 0
    progress.running_sum, progress.running_count = 0.0, 0
    save_detector_training_checkpoint(session, improved, reason="epoch")
    pd.DataFrame(progress.history).to_csv(
        session.checkpoint_directory / LOSS_HISTORY_FILE, index=False
    )


def load_best_detector_weights(session):
    saved_checkpoint = torch.load(
        session.checkpoint_directory / BEST_CHECKPOINT,
        map_location=session.configuration["device"],
        weights_only=True,
    )
    session.detector.model.load_state_dict(saved_checkpoint["model"])
    session.adapter.load_state_dict(saved_checkpoint["adapter"])
    session.detector.model.eval()
    session.adapter.eval()


def fit_detector(detector, adapter, density, parts, arm, config, directory, seed):
    """Resume, train, validate and checkpoint one detector comparison."""
    session = create_detector_training_session(
        detector, adapter, density, parts, arm, config, directory, seed
    )
    restore_detector_training_checkpoint(session)
    while session.progress.epoch < config["epochs"]:
        train_one_detector_epoch(session)
        validation_loss = measure_detector_validation_loss(session)
        record_completed_detector_epoch(session, validation_loss)
        if should_stop_early(
            session.progress.history,
            "val_loss",
            config.get("early_stopping_patience", 0),
        ):
            progress_path = directory / PROGRESS_FILE
            saved_progress = json.loads(progress_path.read_text(encoding="utf-8"))
            saved_progress["reason"] = "early_stopping"
            atomic_json(progress_path, saved_progress)
            break
    load_best_detector_weights(session)


def read_completed_prediction_records(predictions_path, ordered_annotations):
    completed_predictions = []
    if not predictions_path.exists():
        return completed_predictions
    with predictions_path.open("rb+") as stream:
        complete_record_end = 0
        while record_line := stream.readline():
            if not record_line.endswith(b"\n"):
                stream.truncate(complete_record_end)
                break
            completed_predictions.append(json.loads(record_line))
            complete_record_end = stream.tell()
    completed_ids = [prediction["id"] for prediction in completed_predictions]
    expected_ids = [
        annotation["id"]
        for annotation in ordered_annotations[: len(completed_predictions)]
    ]
    if completed_ids != expected_ids:
        raise ValueError(
            "Saved predictions do not match input order; use a new run directory"
        )
    return completed_predictions


def synchronize_inference_device(configuration):
    if configuration["device"].startswith("cuda"):
        torch.cuda.synchronize()


@torch.inference_mode()
def predict_one_image_with_timing(
    detector, adapter, density_estimator, comparison_arm, annotation, configuration
):
    import time

    synchronize_inference_device(configuration)
    start_time = time.perf_counter()
    image = image_tensor(annotation, configuration["size"], configuration["device"])
    synchronize_inference_device(configuration)
    before_density_time = time.perf_counter()
    fog_density = beta_for(comparison_arm, image, annotation, density_estimator)
    synchronize_inference_device(configuration)
    after_density_time = time.perf_counter()
    predictions = detector.predict(adapter(image, fog_density), annotation)
    synchronize_inference_device(configuration)
    end_time = time.perf_counter()
    timing = {
        "total_ms": (end_time - start_time) * 1000,
        "density_ms": (after_density_time - before_density_time) * 1000,
        "detector_adapter_ms": (end_time - after_density_time) * 1000,
    }
    return predictions, float(fog_density.item()), timing


def evaluate_resumable(detector, adapter, density, arm, rows, config, path):
    """Keep valid prediction records, then evaluate only the remaining images."""
    completed_predictions = read_completed_prediction_records(path, rows)
    remaining_annotations = rows[len(completed_predictions) :]
    if remaining_annotations:
        for warmup_index in range(3):
            predict_one_image_with_timing(
                detector, adapter, density, arm, remaining_annotations[0], config
            )
    with path.open("a", encoding="utf-8") as stream:
        for annotation in tqdm(
            remaining_annotations, desc=f"Evaluate {detector.name}/{arm}", mininterval=2
        ):
            timings = []
            for repeat_index in range(config["repeats"]):
                predictions, fog_density, timing = predict_one_image_with_timing(
                    detector, adapter, density, arm, annotation, config
                )
                timings.append(timing)
            prediction_record = {
                "id": annotation["id"],
                "prediction": predictions,
                "beta_input": fog_density,
                "timings": timings,
            }
            stream.write(json.dumps(prediction_record) + "\n")
            stream.flush()
            completed_predictions.append(prediction_record)
    return completed_predictions


def export_trained_model(detector, adapter, directory, identity, config):
    """Write a compact inference artifact without optimizer, scaler or RNG state."""
    payload = {
        "kind": "fog-detector-export-v1",
        **identity,
        "checkpoint_source": config["checkpoints"][detector.name],
        "input_size": config["size"],
        "classes": list(CLASSES),
        "model": {
            key: value.detach().cpu()
            for key, value in detector.model.state_dict().items()
        },
        "adapter": {
            key: value.detach().cpu() for key, value in adapter.state_dict().items()
        },
    }
    save_torch(directory / MODEL_EXPORT_FILE, payload)


def export_saved_checkpoint(name, directory, identity, config):
    """Create the model-only artifact for a run completed by an older engine."""
    checkpoint = directory / BEST_CHECKPOINT
    if not checkpoint.exists() or (directory / MODEL_EXPORT_FILE).exists():
        return
    detector = Detector(
        name, config["checkpoints"][name], config["device"], trainable=False
    )
    adapter = FogAdapter().to(config["device"])
    saved = torch.load(checkpoint, map_location=config["device"], weights_only=True)
    detector.model.load_state_dict(saved["model"])
    adapter.load_state_dict(saved["adapter"])
    export_trained_model(detector, adapter, directory, identity, config)
    del detector, adapter


def read_detector_stop_metadata(directory, configuration):
    progress_path = directory / PROGRESS_FILE
    if not progress_path.exists():
        return {"epochs_trained": 0, "stop_reason": "unknown"}
    progress = json.loads(progress_path.read_text(encoding="utf-8"))
    reason = progress.get("reason", "unknown")
    if reason == "epoch" and progress.get("epoch", 0) >= configuration["epochs"]:
        reason = "epoch_budget"
    return {"epochs_trained": progress.get("epoch", 0), "stop_reason": reason}


def build_run_signature(rows, configuration):
    selected_ids = sorted(row["id"] for row in rows)
    selection_hash = hashlib.sha256("\n".join(selected_ids).encode()).hexdigest()
    return {
        "engine": ENGINE_VERSION,
        "config": configuration,
        "inputs": fingerprint(rows),
        "profile": configuration.get("profile_name", PROFILE_FULL),
        "selection": {
            "schema": 1,
            "ids": selected_ids,
            "ids_sha256": selection_hash,
            "evaluation_split": configuration.get("evaluation_split", "test"),
        },
        "density_sha256": digest(configuration["density_checkpoint"])
        if configuration.get("density_checkpoint")
        else None,
        "source_revision": NOTEBOOK_REVISION,
        "versions": {
            package: importlib.metadata.version(package)
            for package in ("torch", "ultralytics", "transformers", "pycocotools")
        },
    }


def describe_run_identity_changes(previous, current):
    """Explain rejected continuation without dumping the selected image list."""
    changes = []
    for section in ("config", "versions"):
        before, after = previous.get(section, {}), current.get(section, {})
        for name in sorted(set(before) | set(after)):
            if before.get(name) != after.get(name):
                if isinstance(before.get(name), dict) and isinstance(
                    after.get(name), dict
                ):
                    fields = sorted(
                        key
                        for key in set(before[name]) | set(after[name])
                        if before[name].get(key) != after[name].get(key)
                    )
                    changes.append(
                        f"{section}.{name}: changed fields {', '.join(fields)}"
                    )
                else:
                    changes.append(
                        f"{section}.{name}: saved={str(before.get(name, 'not recorded'))[:120]}; requested={str(after.get(name, 'not recorded'))[:120]}"
                    )
    for name in ("engine", "density_sha256"):
        if previous.get(name) != current.get(name):
            changes.append(
                f"{name}: saved={previous.get(name)}; requested={current.get(name)}"
            )
    return "\n".join(changes[:15]) or "Checkpoint engine compatibility differs."


def validate_and_record_run_identity(rows, configuration, output_directory):
    signature = build_run_signature(rows, configuration)
    settings_path = output_directory / RUN_SETTINGS_FILE
    has_checkpoints = any(output_directory.glob(f"*/{LAST_CHECKPOINT}"))
    if has_checkpoints and not settings_path.exists():
        if configuration.get("profile_name") != PROFILE_FULL:
            raise ValueError(
                "Existing checkpoints have no profile identity; choose a new output folder."
            )
    if settings_path.exists():
        previous_signature = json.loads(settings_path.read_text())
        if previous_signature != signature:
            immutable_fields = ("config", "inputs", "density_sha256", "versions")
            compatible = all(
                previous_signature.get(name) == signature.get(name)
                for name in immutable_fields
            )
            is_existing_full = (
                configuration.get("profile_name") == PROFILE_FULL
                and configuration.get("initialization") == "existing_full_run"
            )
            previous_config = previous_signature.get("config", {})
            full_contract_matches = all(
                previous_config.get(key, FULL_RUN_DEFAULTS.get(key))
                == configuration.get(key, FULL_RUN_DEFAULTS.get(key))
                for key in FULL_RUN_IMMUTABLE_KEYS
            )
            if (
                is_existing_full
                and full_contract_matches
                and previous_signature.get("inputs") == signature.get("inputs")
            ):
                print(
                    "Continuing existing full-run checkpoints with recorded settings.",
                    flush=True,
                )
            elif compatible and previous_signature.get("engine") in (
                ENGINE_VERSION - 1,
                ENGINE_VERSION,
            ):
                print(
                    "Migrating compatible checkpoint metadata to the current resumable engine.",
                    flush=True,
                )
            elif previous_signature.get("inputs") != signature.get("inputs"):
                raise ValueError(
                    "The input manifest changed since this run was started. The saved checkpoint belongs to a different dataset version; use a new output folder for the complete-data run. The old checkpoint is preserved."
                )
            else:
                raise ValueError(
                    "Run configuration or package versions changed:\n"
                    + describe_run_identity_changes(previous_signature, signature)
                    + "\nResume with the saved training settings and compatible packages. "
                    "Use analyze for saved results, or retrain=True for a separate fresh run. "
                    "Existing checkpoints were preserved."
                )
    atomic_json(settings_path, signature)
    atomic_json(output_directory / INPUT_MANIFEST_FILE, rows)
    atomic_json(
        output_directory / INPUT_SELECTION_FILE,
        signature["selection"],
    )


def prepare_density_estimator_for_seed(
    dataset_partitions, configuration, output_directory, seed
):
    if ARM_DENSITY not in configuration["arms"]:
        return None
    if configuration.get("density_checkpoint"):
        return load_density_predictor(
            configuration["density_checkpoint"], configuration["device"]
        )
    density_directory = output_directory / f"{DENSITY_RUN_PREFIX}-{seed}"
    density_estimator = fit_density(
        dataset_partitions, configuration, density_directory, seed
    )
    predictions_path = density_directory / "test-predictions.csv"
    if not predictions_path.exists():
        density_predictions = []
        with torch.inference_mode():
            for annotation in tqdm(
                dataset_partitions["density_evaluation"],
                desc="Held-out density quality",
                mininterval=2,
            ):
                image = image_tensor(
                    annotation, configuration["size"], configuration["device"]
                )
                estimate = float(density_estimator(image).item())
                density_predictions.append(
                    {
                        "id": annotation["id"],
                        "dataset": annotation["dataset"],
                        "target_beta": annotation["beta"],
                        "predicted_beta": estimate,
                    }
                )
        pd.DataFrame(
            density_predictions,
            columns=["id", "dataset", "target_beta", "predicted_beta"],
        ).to_csv(predictions_path, index=False)
    return density_estimator


def evaluate_detector_comparison(
    detector,
    adapter,
    density_estimator,
    dataset_partitions,
    configuration,
    directory,
    identity,
    trainable_parameter_count,
):
    summaries = []
    evaluation_annotations = dataset_partitions["evaluation"]
    for dataset_name in sorted(
        {annotation["dataset"] for annotation in evaluation_annotations}
    ):
        test_annotations = sorted(
            [
                annotation
                for annotation in evaluation_annotations
                if annotation["dataset"] == dataset_name
            ],
            key=lambda annotation: annotation["id"],
        )
        uses_cuda = configuration["device"].startswith("cuda")
        if uses_cuda:
            torch.cuda.reset_peak_memory_stats()
        predictions = evaluate_resumable(
            detector,
            adapter,
            density_estimator,
            identity["arm"],
            test_annotations,
            configuration,
            directory / f"{dataset_name}-predictions.jsonl",
        )
        peak_gpu_megabytes = (
            torch.cuda.max_memory_allocated() / 2**20 if uses_cuda else None
        )
        for metrics in summarize(test_annotations, predictions):
            summaries.append(
                {
                    **identity,
                    "dataset": dataset_name,
                    **metrics,
                    "peak_gpu_mb": peak_gpu_megabytes,
                    "trainable_detector_parameters": trainable_parameter_count,
                    "trained_checkpoint_mb": Path(
                        identity.get("source_checkpoint", directory / BEST_CHECKPOINT)
                    )
                    .stat()
                    .st_size
                    / 2**20,
                }
            )
        curves = []
        for condition in ("clear", "foggy"):
            indices = [
                index
                for index, row in enumerate(test_annotations)
                if row["beta"] is not None
                and (row["beta"] == 0 if condition == "clear" else row["beta"] > 0)
            ]
            if indices:
                curve = coco_metrics(
                    [test_annotations[index] for index in indices],
                    [predictions[index] for index in indices],
                    include_curve=True,
                )
                curves.append(
                    {
                        **identity,
                        "dataset": dataset_name,
                        "condition": condition,
                        "precision": curve["precision_curve"],
                        "recall": curve["recall_curve"],
                    }
                )
        atomic_json(directory / f"{dataset_name}-precision-recall.json", curves)
    pd.DataFrame(summaries).to_csv(directory / RESULTS_FILE, index=False)


def validate_matched_evaluation(rows, source_rows, split, max_pairs=None):
    """Require the same labeled scenes at each fog level, held out from fitting."""
    selected = [row for row in rows if row["split"] == split]
    if not selected:
        raise ValueError(f"No matched evaluation images in split {split}.")
    pairs = {}
    for row in selected:
        if not row.get("detection_labeled") or not row.get("annotation_source"):
            raise ValueError("Matched evaluation requires gtFine detection labels.")
        if not Path(row["annotation_source"]).is_file():
            raise FileNotFoundError(row["annotation_source"])
        if row.get("beta") is None or not row.get("pair_id"):
            raise ValueError("Matched evaluation requires known beta and pair_id.")
        pairs.setdefault(row["pair_id"], []).append(row)
    expected = set(DENSITY_BIN_UPPER_BOUNDS)
    for pair, variants in pairs.items():
        if len(variants) != len(expected) or {r["beta"] for r in variants} != expected:
            raise ValueError(f"Incomplete or duplicate clear/fog variants: {pair}")
        reference = variants[0]
        for row in variants:
            if any(
                row[key] != reference[key]
                for key in ("boxes", "labels", "iscrowd", "width", "height", "group")
            ):
                raise ValueError(f"Mismatched ground truth: {pair}")
    fitted_groups = {
        (r["dataset"], r["group"])
        for r in source_rows
        if r["split"] in ("detector_train", "density_train")
        or (split == "test" and r["split"] == "val")
    }
    if any((r["dataset"], r["group"]) in fitted_groups for r in selected):
        raise ValueError("Evaluation scenes overlap checkpoint fitting/tuning scenes.")
    if max_pairs is not None:
        if (
            not isinstance(max_pairs, int)
            or isinstance(max_pairs, bool)
            or max_pairs < 1
        ):
            raise ValueError("max_pairs must be a positive integer or None.")
        chosen = set(
            sorted(pairs, key=lambda pair: hashlib.sha256(pair.encode()).hexdigest())[
                :max_pairs
            ]
        )
        selected = [row for row in selected if row["pair_id"] in chosen]
    return sorted(selected, key=lambda row: row["id"])


def run_checkpoint_evaluation(rows, configuration):
    """Evaluate saved best weights; source runs are read-only and never trained."""

    source = Path(configuration["evaluation_source"]).resolve()
    output = Path(configuration["output"]).resolve()
    if source == output or source in output.parents or output in source.parents:
        raise ValueError("Evaluation output must be separate from the training run.")
    output.mkdir(parents=True, exist_ok=True)
    with (
        FileLock(str(source / RUN_LOCK_FILE), timeout=0),
        FileLock(str(output / RUN_LOCK_FILE), timeout=0),
    ):
        source_settings = json.loads((source / RUN_SETTINGS_FILE).read_text())
        source_config = source_settings["config"]
        if configuration["size"] != source_config["size"]:
            raise ValueError("Use the source checkpoint's image size for evaluation.")
        source_rows = json.loads((source / INPUT_MANIFEST_FILE).read_text())
        evaluation_rows = validate_matched_evaluation(
            rows,
            source_rows,
            configuration["evaluation_split"],
            configuration.get("evaluation_max_pairs"),
        )
        sources = []
        for seed in configuration["seeds"]:
            for model in configuration["checkpoints"]:
                for arm in configuration["arms"]:
                    checkpoint = source / f"{model}-{arm}-{seed}" / BEST_CHECKPOINT
                    if not checkpoint.is_file():
                        raise FileNotFoundError(f"Missing trained arm: {checkpoint}")
                    if (
                        arm not in source_config["arms"]
                        or seed not in source_config["seeds"]
                    ):
                        raise ValueError("Arm/seed was not trained in the source run.")
                    sources.append((model, arm, seed, checkpoint))
            if ARM_DENSITY in configuration["arms"]:
                density_path = Path(
                    source_config.get("density_checkpoint")
                    or source / f"{DENSITY_RUN_PREFIX}-{seed}" / BEST_CHECKPOINT
                )
                if not density_path.is_file():
                    raise FileNotFoundError(density_path)
        identity_config = dict(
            configuration,
            initialization="pretrained",
            source_training_config=source_config,
            checkpoint_sources={
                str(path): [path.stat().st_size, path.stat().st_mtime_ns]
                for _, _, _, path in sources
            },
            evaluation_scope="matched_cityscapes_domain_shift",
        )
        density_paths = (
            [Path(source_config["density_checkpoint"])]
            if source_config.get("density_checkpoint")
            else [
                source / f"{DENSITY_RUN_PREFIX}-{seed}" / BEST_CHECKPOINT
                for seed in configuration["seeds"]
            ]
        )
        identity_config["density_sources"] = {
            str(path): [path.stat().st_size, path.stat().st_mtime_ns]
            for path in density_paths
        }
        identity_config["annotation_sources"] = {
            str(path): [path.stat().st_size, path.stat().st_mtime_ns]
            for path in {Path(row["annotation_source"]) for row in evaluation_rows}
        }
        validate_and_record_run_identity(evaluation_rows, identity_config, output)
        for model, arm, seed, checkpoint in sources:
            evaluate_saved_comparison(
                model, arm, seed, checkpoint, evaluation_rows, configuration, output
            )
        return export_analytics(output, evaluation_rows), pd.read_csv(
            output / STATUS_TABLE_FILE
        )


def evaluate_saved_comparison(model, arm, seed, checkpoint, rows, config, output):
    directory = output / f"{model}-{arm}-{seed}"
    directory.mkdir(exist_ok=True)
    status_path = directory / STATUS_FILE
    if (
        status_path.exists()
        and json.loads(status_path.read_text())["status"] == "complete"
    ):
        return
    identity = {
        "model": model,
        "arm": arm,
        "seed": seed,
        "profile": config.get("profile_name", PROFILE_FULL),
        "evaluation_split": config["evaluation_split"],
        "source_checkpoint": str(checkpoint),
        "evaluation_scope": "matched_cityscapes_domain_shift",
    }
    detector = adapter = density = None
    try:
        detector = Detector(
            model, config["checkpoints"][model], config["device"], trainable=True
        )
        adapter = FogAdapter().to(config["device"])
        saved = torch.load(checkpoint, map_location=config["device"], weights_only=True)
        source_config = json.loads(
            (checkpoint.parent.parent / RUN_SETTINGS_FILE).read_text()
        )["config"]
        identity["planned_epochs"] = source_config["epochs"]
        identity["best_epoch"] = saved.get("epoch")
        detector.model.load_state_dict(saved["model"])
        adapter.load_state_dict(saved["adapter"])
        detector.model.eval()
        adapter.eval()
        if arm == ARM_DENSITY:
            density_checkpoint = source_config.get("density_checkpoint") or (
                Path(config["evaluation_source"])
                / f"{DENSITY_RUN_PREFIX}-{seed}"
                / BEST_CHECKPOINT
            )
            density = load_density_predictor(density_checkpoint, config["device"])
        identity.update(read_detector_stop_metadata(checkpoint.parent, source_config))
        atomic_json(status_path, {**identity, "status": "evaluating"})
        evaluate_detector_comparison(
            detector,
            adapter,
            density,
            {"evaluation": rows},
            config,
            directory,
            identity,
            sum(p.numel() for p in detector.model.parameters()),
        )
        normal_path = checkpoint.parent / RESULTS_FILE
        if normal_path.exists():
            normal = pd.read_csv(normal_path, dtype={"density": str})
            normal = normal[normal.dataset == "nuscenes"].copy()
            normal["evaluation_scope"] = "source_normal_use_fog_unknown"
            measured = pd.read_csv(directory / RESULTS_FILE, dtype={"density": str})
            pd.concat([measured, normal], ignore_index=True).to_csv(
                directory / RESULTS_FILE, index=False
            )
        atomic_json(status_path, {**identity, "status": "complete"})
    except BaseException as error:
        atomic_json(
            status_path,
            {
                **identity,
                "status": "interrupted"
                if isinstance(error, KeyboardInterrupt)
                else "error",
                "reason": f"{type(error).__name__}: {error}",
            },
        )
        raise
    finally:
        del detector, adapter, density
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def run_detector_comparison(
    model_name,
    comparison_arm,
    seed,
    density_estimator,
    dataset_partitions,
    configuration,
    output_directory,
):
    directory = output_directory / f"{model_name}-{comparison_arm}-{seed}"
    directory.mkdir(exist_ok=True)
    identity = detector_comparison_identity(
        model_name, comparison_arm, seed, configuration
    )
    status_path = directory / STATUS_FILE
    if detector_comparison_is_complete(status_path):
        export_saved_checkpoint(model_name, directory, identity, configuration)
        return json.loads(status_path.read_text())
    detector, adapter = create_detector_comparison_models(
        model_name, seed, configuration
    )
    try:
        run_detector_training_and_evaluation(
            detector,
            adapter,
            density_estimator,
            dataset_partitions,
            configuration,
            directory,
            identity,
            status_path,
        )
    except BaseException as error:
        record_detector_comparison_error(status_path, identity, error)
        raise
    finally:
        release_detector_comparison_models(detector, adapter)
    return json.loads(status_path.read_text())


def detector_comparison_identity(model_name, comparison_arm, seed, configuration):
    return {
        "model": model_name,
        "arm": comparison_arm,
        "seed": seed,
        "profile": configuration.get("profile_name", PROFILE_FULL),
        "evaluation_split": configuration.get("evaluation_split", "test"),
        "planned_epochs": configuration["epochs"],
    }


def detector_comparison_is_complete(status_path):
    return (
        status_path.exists()
        and json.loads(status_path.read_text()).get("status") == "complete"
    )


def create_detector_comparison_models(model_name, seed, configuration):
    seed_all(seed)
    detector = Detector(
        model_name,
        configuration["checkpoints"][model_name],
        configuration["device"],
        trainable=True,
    )
    seed_all(seed)
    adapter = FogAdapter().to(configuration["device"])
    return detector, adapter


def run_detector_training_and_evaluation(
    detector,
    adapter,
    density_estimator,
    dataset_partitions,
    configuration,
    directory,
    identity,
    status_path,
):
    trainable_parameters = record_detector_training_started(
        status_path, identity, detector
    )
    fit_detector(
        detector,
        adapter,
        density_estimator,
        dataset_partitions,
        identity["arm"],
        configuration,
        directory,
        identity["seed"],
    )
    identity.update(read_detector_stop_metadata(directory, configuration))
    export_trained_model(detector, adapter, directory, identity, configuration)
    atomic_json(status_path, {**identity, "status": "evaluating"})
    evaluate_detector_comparison(
        detector,
        adapter,
        density_estimator,
        dataset_partitions,
        configuration,
        directory,
        identity,
        trainable_parameters,
    )
    atomic_json(status_path, {**identity, "status": "complete"})


def record_detector_training_started(status_path, identity, detector):
    trainable_parameters = sum(
        parameter.numel()
        for parameter in detector.model.parameters()
        if parameter.requires_grad
    )
    atomic_json(
        status_path,
        {
            **identity,
            "status": "training",
            "trainable_detector_parameters": trainable_parameters,
        },
    )
    return trainable_parameters


def record_detector_comparison_error(status_path, identity, error):
    status = "interrupted" if isinstance(error, KeyboardInterrupt) else "error"
    atomic_json(
        status_path,
        {**identity, "status": status, "reason": f"{type(error).__name__}: {error}"},
    )


def release_detector_comparison_models(detector, adapter):
    del detector, adapter
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def run_training(rows, config, reset_setup=True):
    """Validate identity, resume each comparison, and produce the study reports."""
    require_ready_preflight(config, rows)
    output_directory = Path(config["output"])
    validate_setup_overwrite(config)
    output_directory.mkdir(parents=True, exist_ok=True)
    with FileLock(str(output_directory / RUN_LOCK_FILE), timeout=0):
        clear_setup_output_if_requested(config, output_directory, reset_setup)
        validate_and_record_run_identity(rows, config, output_directory)
        dataset_partitions = partitions(rows, config)
        statuses = run_comparison_matrix(dataset_partitions, config, output_directory)
        return export_training_results(output_directory, rows, statuses)


def require_ready_preflight(config, rows):
    checks = preflight(config, rows)
    if checks.ready.all():
        return
    failures = checks.loc[~checks.ready, ["check", "detail"]].to_string(index=False)
    raise RuntimeError("Preflight failed:\n" + failures)


def validate_setup_overwrite(config):
    overwrite = config.get("overwrite_existing", False)
    if overwrite and config.get("profile_name") != PROFILE_SETUP:
        raise ValueError("overwrite_existing is only allowed for the setup profile.")


def clear_setup_output_if_requested(config, output_directory, reset_setup):
    if not config.get("overwrite_existing", False) or not reset_setup:
        return
    for child in output_directory.iterdir():
        if child.name in (RUN_LOCK_FILE, SUBSET_AUDIT_FILE):
            continue
        if not child.resolve().is_relative_to(output_directory.resolve()):
            raise ValueError("Setup cleanup target escapes its output folder.")
        if child.is_dir():
            shutil.rmtree(child)
        else:
            child.unlink()


def run_comparison_matrix(dataset_partitions, config, output_directory):
    statuses = []
    model_names = list(config["checkpoints"])
    for seed_index, seed in enumerate(config["seeds"]):
        density = prepare_density_estimator_for_seed(
            dataset_partitions, config, output_directory, seed
        )
        model_order = (
            model_names[seed_index % len(model_names) :]
            + model_names[: seed_index % len(model_names)]
        )
        for model_name in model_order:
            for arm in config["arms"]:
                statuses.append(
                    run_detector_comparison(
                        model_name,
                        arm,
                        seed,
                        density,
                        dataset_partitions,
                        config,
                        output_directory,
                    )
                )
    return statuses


def export_training_results(output_directory, rows, statuses):

    return export_analytics(output_directory, rows), pd.DataFrame(statuses)

#### Notebook setup and high-level workflow steps

In [ ]:
# fmt: off
#@title Supporting code: Notebook setup and high-level workflow steps
# fmt: on
"""Storage, data loading, execution and presentation for this notebook."""


def relative_path(value):
    value = str(value)
    path = PurePosixPath(value)
    if (
        not value
        or path.is_absolute()
        or ".." in path.parts
        or "\\" in value
        or ":" in value
    ):
        raise ValueError(f"Expected a nonempty relative path: {value!r}")
    return Path(value)


def location_path(location):
    if location["kind"] == "drive":
        return (
            Path(SETTINGS["drive_mount"])
            / relative_path(SETTINGS["drive_subdir"])
            / relative_path(location["path"])
        )
    if location["kind"] == "path":
        path = Path(location["path"])
        if not path.is_absolute():
            raise ValueError("Filesystem locations must be absolute paths.")
        return path
    raise ValueError("Location kind must be 'drive' or 'path'.")


def single_name(value):
    path = relative_path(value)
    if len(path.parts) != 1 or value in (".", ".."):
        raise ValueError(f"Expected a single filename or directory name: {value!r}")
    return value


def validate_study_settings():
    global ROOT
    SETTINGS["action"] = str(SETTINGS["action"]).strip().lower()
    if SETTINGS["action"] not in {
        ACTION_TRAIN,
        ACTION_PREFLIGHT,
        ACTION_ANALYZE,
        ACTION_EVALUATE,
    }:
        raise ValueError("Unknown action.")
    if not isinstance(SETTINGS.get("retrain", False), bool):
        raise ValueError("retrain must be True or False.")
    if SETTINGS.get("retrain", False) and SETTINGS["action"] != ACTION_TRAIN:
        raise ValueError("retrain=True requires action='train'.")
    profile = SETTINGS.get("profile", {})
    profile_name = profile.get("name")
    if profile_name not in PROFILE_NAMES:
        raise ValueError(f"profile.name must be one of {PROFILE_NAMES}")
    if profile.get("initialization") not in {"pretrained", "existing_full_run"}:
        raise ValueError("profile.initialization is invalid")
    if (
        profile_name == PROFILE_FULL
        and profile.get("initialization") != "existing_full_run"
    ):
        raise ValueError("The full profile must identify the existing full run.")
    if profile_name != PROFILE_FULL and profile.get("initialization") != "pretrained":
        raise ValueError("Partial profiles must start from pretrained weights.")
    predictor = SETTINGS["density_predictor"]
    single_name(predictor["source_run"])
    if (
        not isinstance(predictor["seed"], int)
        or isinstance(predictor["seed"], bool)
        or predictor["seed"] < 0
    ):
        raise ValueError("density_predictor.seed must be a nonnegative integer.")
    data_settings = SETTINGS["data"]
    if data_settings["mode"] not in {"zip", "directory"}:
        raise ValueError("Data mode must be zip or directory.")
    if data_settings["minimum_free_gib"] < 0:
        raise ValueError("minimum_free_gib cannot be negative.")
    single_name(data_settings["archive"])
    single_name(SETTINGS["output"]["run_name"])
    single_name(SETTINGS["output"]["export_name"])
    single_name(SETTINGS["output"]["packages_file"])
    ROOT = Path(SETTINGS["runtime_root"])
    if not ROOT.is_absolute() or not Path(SETTINGS["drive_mount"]).is_absolute():
        raise ValueError("runtime_root and drive_mount must be absolute.")
    if data_settings["mode"] == "directory" and data_settings["url"]:
        raise ValueError("A data URL is only supported in zip mode.")
    if data_settings["low_disk"] and (
        data_settings["location"]["kind"] != "drive" or data_settings["url"]
    ):
        raise ValueError("Low-disk mode requires data in Drive with no URL.")


def connect_and_resolve_study_storage():
    global DATA_ROOT, RESOLVED_PATHS, TRAINING_CONFIGURATION
    data_settings = SETTINGS["data"]
    # Resolve paths; authentication already ran before setup.
    data_location = location_path(data_settings["location"])
    output_location = location_path(SETTINGS["output"]["location"])
    if any(
        loc["kind"] == "drive"
        for loc in (data_settings["location"], SETTINGS["output"]["location"])
    ):
        drive_root = Path(SETTINGS["drive_mount"]) / SETTINGS["drive_subdir"]
        if not drive_root.is_dir():
            raise RuntimeError(
                "Google Drive is disconnected. Rerun the connection cell near the top."
            )
    ROOT.mkdir(parents=True, exist_ok=True)
    os.chdir(ROOT)
    DATA_ROOT = data_location if data_settings["mode"] == "directory" else ROOT
    RESOLVED_PATHS = {
        "data_location": data_location,
        "data_root": DATA_ROOT,
        "archive": data_location / data_settings["archive"],
        "output_root": output_location,
        "run": output_location / SETTINGS["output"]["run_name"],
        "export": output_location / SETTINGS["output"]["export_name"],
        "marker": ROOT / relative_path(data_settings["extraction_marker"]),
        "packages": ROOT / SETTINGS["output"]["packages_file"],
        "city_images": DATA_ROOT / relative_path(data_settings["city_images"]),
        "nuscenes": DATA_ROOT / relative_path(data_settings["nuscenes"]),
    }
    RESOLVED_PATHS["run"].mkdir(parents=True, exist_ok=True)
    TRAINING_CONFIGURATION = dict(SETTINGS["training"])


def apply_subset_execution_settings():
    global DATA_ROOT, SETTINGS
    output_location = RESOLVED_PATHS["output_root"]
    test_settings = SETTINGS.get("test", {})
    if test_settings.get("enabled"):
        smoke = dict(PROFILE_SETTINGS[PROFILE_SETUP])
        smoke["name"] = PROFILE_SETUP
        smoke["run_suffix"] = f"-{smoke['run_name']}"
        smoke["initialization"] = "pretrained"
        smoke["models"] = test_settings.get("models", smoke["models"])
        smoke["seed"] = test_settings.get("seed", smoke["seed"])
        smoke["epochs"] = test_settings.get("epochs", smoke["epochs"])
        smoke["size"] = test_settings.get("image_size", smoke["size"])
        images_per_stratum = test_settings.get("images_per_stratum", 4)
        smoke["training_image_cap"] = max(6, images_per_stratum * 6)
        smoke["tuning_image_cap"] = max(3, images_per_stratum * 3)
        SETTINGS["profile"] = smoke
    profile = SETTINGS["profile"]
    profile_name = profile["name"]
    if profile_name != PROFILE_FULL:
        single_name(profile["run_suffix"])
        RESOLVED_PATHS["run"] = output_location / SETTINGS["output"]["run_name"]
        RESOLVED_PATHS["export"] = output_location / SETTINGS["output"]["export_name"]
        RESOLVED_PATHS["run"].mkdir(parents=True, exist_ok=True)
        TRAINING_CONFIGURATION.update(
            epochs=profile["epochs"],
            density_epochs=profile.get(
                "density_epochs", TRAINING_CONFIGURATION["density_epochs"]
            ),
            size=profile["size"],
            save_every=profile["save_every"],
            seeds=[profile["seed"]],
            early_stopping_patience=profile["early_stopping_patience"],
            profile_name=profile_name,
            initialization="pretrained",
            evaluation_split=profile["evaluation_split"],
            density_sampling=profile.get("density_sampling", "natural"),
            overwrite_existing=profile.get("overwrite_existing", False),
        )
        TRAINING_CONFIGURATION["profile_scope"] = dict(profile)
        TRAINING_CONFIGURATION["test_scope"] = {
            "enabled": True,
            "profile": profile_name,
            "scientific_results": profile_name != PROFILE_SETUP,
        }
        RESOLVED_PATHS["data_root"] = ROOT / ("profile-" + profile_name)
        RESOLVED_PATHS["data_root"].mkdir(parents=True, exist_ok=True)
        DATA_ROOT = RESOLVED_PATHS["data_root"]
    else:
        TRAINING_CONFIGURATION.update(
            profile_name=PROFILE_FULL,
            initialization="existing_full_run",
            evaluation_split="test",
            early_stopping_patience=0,
            density_sampling=profile.get("density_sampling", "natural"),
        )


def resolve_manifest_and_model_paths():
    data_settings = SETTINGS["data"]
    selected_models = SETTINGS["profile"]["models"]
    TRAINING_CONFIGURATION.update(
        {
            key: str(DATA_ROOT / relative_path(value))
            for key, value in data_settings["manifests"].items()
        }
    )
    TRAINING_CONFIGURATION["output"] = str(RESOLVED_PATHS["run"])
    TRAINING_CONFIGURATION["checkpoints"] = {
        name: value if name == DETR_MODEL else str(ROOT / Path(value))
        for name, value in SETTINGS["models"].items()
        if name in selected_models
    }
    if TRAINING_CONFIGURATION["density_checkpoint"]:
        TRAINING_CONFIGURATION["density_checkpoint"] = str(
            ROOT / Path(TRAINING_CONFIGURATION["density_checkpoint"])
        )


def resolve_study_configuration(settings):
    """Validate settings, connect storage and resolve one experiment configuration."""
    global SETTINGS
    SETTINGS = settings
    validate_study_settings()
    connect_and_resolve_study_storage()
    saved_configuration = restore_existing_profile_settings()
    apply_training_profile(saved_configuration)
    resolve_default_export_path()
    allocate_fresh_retrain_output()
    configure_density_predictor_checkpoint()
    configure_checkpoint_evaluation()
    return TRAINING_CONFIGURATION, RESOLVED_PATHS


def apply_training_profile(saved_configuration):
    """Resolve new settings unless this output folder owns a saved experiment."""
    apply_subset_execution_settings()
    resolve_manifest_and_model_paths()
    if saved_configuration is None:
        return
    TRAINING_CONFIGURATION.clear()
    TRAINING_CONFIGURATION.update(saved_configuration)


def resolve_default_export_path():
    """Keep a default report folder beside the selected run folder."""
    expected = SETTINGS["profile"]["run_name"] + "-results"
    if SETTINGS["output"]["export_name"] != expected:
        return
    run = RESOLVED_PATHS["run"]
    RESOLVED_PATHS["export"] = run.with_name(run.name + "-results")


def allocate_fresh_retrain_output():
    """Give retraining its own folder so prior checkpoints remain recoverable."""
    if (
        not SETTINGS.get("retrain", False)
        or SETTINGS["profile"]["name"] == PROFILE_SETUP
    ):
        return
    run = RESOLVED_PATHS["run"]
    output = run.with_name(run.name + "-retrain-" + uuid4().hex[:12])
    output.mkdir(parents=True, exist_ok=False)
    TRAINING_CONFIGURATION.update(
        output=str(output), initialization="pretrained", density_checkpoint=None
    )
    RESOLVED_PATHS["run"] = output
    RESOLVED_PATHS["export"] = output.with_name(output.name + "-results")
    print("Fresh training requested. Previous run preserved. New output:", output)


def configure_checkpoint_evaluation():
    """Switch to a separate output only for matched checkpoint evaluation."""
    if SETTINGS["action"] != ACTION_EVALUATE:
        return
    options = SETTINGS["evaluation"]
    source = Path(options["source_run"] or TRAINING_CONFIGURATION["output"])
    output = RESOLVED_PATHS["output_root"] / single_name(options["output_name"])
    if source.resolve() == output.resolve():
        raise ValueError("Evaluation must use a separate output_name.")
    TRAINING_CONFIGURATION.update(
        evaluation_source=str(source),
        output=str(output),
        evaluation_split=options["split"],
        overwrite_existing=False,
        evaluation_max_pairs=options.get("max_pairs"),
        evaluation_cityscapes_root=str(Path(options["cityscapes_root"]).resolve()),
    )
    RESOLVED_PATHS["run"] = output
    RESOLVED_PATHS["export"] = output.with_name(output.name + "-results")


def configure_density_predictor_checkpoint():
    """Load a completed estimator for new detector profiles; preserve full resumes."""
    options = SETTINGS["density_predictor"]
    configured = str(options.get("checkpoint", "")).strip()
    if configured:
        checkpoint = Path(configured)
        if not checkpoint.is_absolute():
            checkpoint = ROOT / checkpoint
    elif (
        SETTINGS["profile"]["name"] == PROFILE_FULL
        and (RESOLVED_PATHS["run"] / RUN_SETTINGS_FILE).is_file()
    ):
        return
    else:
        checkpoint = (
            RESOLVED_PATHS["output_root"]
            / options["source_run"]
            / f"{DENSITY_RUN_PREFIX}-{options['seed']}"
            / BEST_CHECKPOINT
        )
    TRAINING_CONFIGURATION["density_checkpoint"] = str(checkpoint.resolve())


def restore_existing_profile_settings():
    """An existing output folder defines the experiment; new defaults do not."""
    path = RESOLVED_PATHS["run"] / RUN_SETTINGS_FILE
    if (
        SETTINGS["action"] != ACTION_TRAIN
        or SETTINGS.get("retrain", False)
        or SETTINGS.get("test", {}).get("enabled", False)
        or SETTINGS["profile"]["name"] == PROFILE_SETUP
        or not path.exists()
    ):
        return None
    recorded = json.loads(path.read_text(encoding="utf-8"))["config"]
    if recorded.get("profile_name", PROFILE_FULL) != SETTINGS["profile"]["name"]:
        raise ValueError("Saved output belongs to another profile.")
    SETTINGS["profile"].update(recorded.get("profile_scope", {}))
    SETTINGS["profile"]["density_sampling"] = recorded.get(
        "density_sampling", "natural"
    )
    SETTINGS["profile"]["overwrite_existing"] = False
    recorded["output"] = str(RESOLVED_PATHS["run"])
    if SETTINGS["profile"]["name"] == PROFILE_FULL:
        recorded.setdefault("profile_name", PROFILE_FULL)
        recorded.setdefault("initialization", "existing_full_run")
    print("Using this run's recorded training settings (including density sampling).")
    return recorded


def should_reuse_completed_training(configuration):
    """Check saved experiment statuses without opening images or model weights."""
    if (
        SETTINGS["action"] != ACTION_TRAIN
        or SETTINGS["profile"]["name"] == PROFILE_SETUP
        or SETTINGS.get("retrain", False)
    ):
        return False
    output = Path(configuration["output"])
    if not (output / RUN_SETTINGS_FILE).is_file():
        return False
    directories = [
        output / f"{model}-{arm}-{seed}"
        for model in configuration["checkpoints"]
        for arm in configuration["arms"]
        for seed in configuration["seeds"]
    ]
    return bool(directories) and all(
        (directory / RESULTS_FILE).is_file()
        and (directory / STATUS_FILE).is_file()
        and json.loads((directory / STATUS_FILE).read_text()).get("status")
        == "complete"
        for directory in directories
    )


def check_free_space(required=0):
    free = shutil.disk_usage(ROOT).free
    reserve = int(SETTINGS["data"]["minimum_free_gib"] * 2**30)
    if required + reserve > free:
        raise OSError(
            f"Need {(required + reserve) / 2**30:.2f} GiB free; available {free / 2**30:.2f} GiB."
        )


def select_subset(annotations):
    profile = SETTINGS["profile"]
    eligible = [
        annotation
        for annotation in annotations
        if not (
            annotation["split"] in ("detector_train", "val")
            and (
                not annotation.get("detection_labeled", True)
                or any(annotation.get("iscrowd", []))
            )
            and annotation["beta"] is None
        )
    ]
    limits = {
        "detector_train": profile["training_image_cap"],
        "val": profile["tuning_image_cap"],
    }
    if "density_image_cap" in profile:
        limits["density_train"] = profile["density_image_cap"]
    selected = select_profile_rows(eligible, profile["name"], profile["seed"], limits)
    if not selected:
        raise ValueError("No eligible subset images in this manifest.")
    return selected


def prepare_subset():
    from pathlib import PurePosixPath

    data_settings = SETTINGS["data"]
    destination_root = RESOLVED_PATHS["data_root"].resolve()
    archive = None
    if data_settings["mode"] == "zip":
        if data_settings["url"]:
            raise ValueError(
                "Subset ZIP mode requires an existing archive, not a URL download."
            )
        if not RESOLVED_PATHS["archive"].is_file():
            raise FileNotFoundError(RESOLVED_PATHS["archive"])
        archive = zipfile.ZipFile(RESOLVED_PATHS["archive"])
    manifest_annotations, images = {}, set()
    try:
        for key, relative in data_settings["manifests"].items():
            if key == "city_manifest" and TRAINING_CONFIGURATION.get(
                "density_checkpoint"
            ):
                continue
            relative_path(relative)
            manifest_text = (
                archive.read(relative).decode("utf-8")
                if archive
                else (RESOLVED_PATHS["data_location"] / relative).read_text(
                    encoding="utf-8"
                )
            )
            annotations = select_subset(json.loads(manifest_text))
            for annotation in annotations:
                image = str(annotation["image"])
                if archive is None:
                    source_image = (
                        RESOLVED_PATHS["data_location"] / relative
                    ).parent / image
                    source_image = source_image.resolve()
                    archive_member_name = source_image.relative_to(
                        RESOLVED_PATHS["data_location"].resolve()
                    ).as_posix()
                    annotation["image"] = os.path.relpath(
                        archive_member_name, str(PurePosixPath(relative).parent)
                    ).replace("\\", "/")
                    images.add(archive_member_name)
                    continue
                # Portable manifests must refer to files within the data bundle.
                if PurePosixPath(image).is_absolute() or "\\" in image or ":" in image:
                    raise ValueError(f"Nonportable manifest image: {image}")
                manifest_parent = PurePosixPath(relative).parent
                path_components = []
                for part in (manifest_parent / image).parts:
                    if part == "..":
                        if not path_components:
                            raise ValueError(f"Image escapes dataset root: {image}")
                        path_components.pop()
                    elif part != ".":
                        path_components.append(part)
                archive_member_name = "/".join(path_components)
                if not archive_member_name.startswith(
                    tuple(data_settings["allowed_archive_roots"])
                ):
                    raise ValueError(f"Unexpected dataset image: {archive_member_name}")
                images.add(archive_member_name)
            manifest_annotations[relative] = annotations
        check_free_space(
            sum(archive.getinfo(name).file_size for name in images)
            if archive
            else sum(
                (RESOLVED_PATHS["data_location"] / name).stat().st_size
                for name in images
            )
        )
        for archive_member_name in sorted(images):
            target = destination_root / archive_member_name
            target.parent.mkdir(parents=True, exist_ok=True)
            if archive:
                info = archive.getinfo(archive_member_name)
                if stat.S_ISLNK(info.external_attr >> 16):
                    raise ValueError(
                        f"Archive image is a symlink: {archive_member_name}"
                    )
                with archive.open(info) as source, target.open("wb") as destination:
                    shutil.copyfileobj(source, destination)
            else:
                shutil.copy2(
                    RESOLVED_PATHS["data_location"] / archive_member_name, target
                )
        for relative, annotations in manifest_annotations.items():
            manifest = destination_root / relative
            manifest.parent.mkdir(parents=True, exist_ok=True)
            manifest.write_text(json.dumps(annotations), encoding="utf-8")
        selected_ids = sorted(
            annotation["id"]
            for annotations in manifest_annotations.values()
            for annotation in annotations
        )
        report = {
            "profile": SETTINGS["profile"],
            "selected_ids": selected_ids,
            "scope": {
                "training_image_cap": SETTINGS["profile"]["training_image_cap"],
                "tuning_image_cap": SETTINGS["profile"]["tuning_image_cap"],
            },
            "images": len(images),
            "records_per_manifest": {
                name: len(annotations)
                for name, annotations in manifest_annotations.items()
            },
            "scientific_results": SETTINGS["profile"]["name"] != PROFILE_SETUP,
        }
        (RESOLVED_PATHS["run"] / SUBSET_AUDIT_FILE).write_text(
            json.dumps(report, indent=2)
        )
        print(
            f"Prepared {report['images']} images for {SETTINGS['profile']['name']}; subset audit saved."
        )
    finally:
        if archive:
            archive.close()


def prepare_fog_data():
    data_settings = SETTINGS["data"]
    if should_reuse_completed_training(TRAINING_CONFIGURATION):
        print(
            "Training already complete. Reusing saved results; data preparation skipped."
        )
        return
    if SETTINGS["action"] in {ACTION_ANALYZE, ACTION_EVALUATE}:
        print("Analysis uses saved run outputs; dataset extraction is skipped.")
        return
    if SETTINGS["profile"]["name"] != PROFILE_FULL:
        prepare_subset()
        return
    if data_settings["mode"] == "directory":
        for name in data_settings["manifests"].values():
            path = RESOLVED_PATHS["data_root"] / relative_path(name)
            if not path.is_file():
                raise FileNotFoundError(f"Missing prepared manifest: {path}")
        check_free_space()
        return
    dataset_archive = find_or_download_dataset_archive()
    validate_and_extract_full_dataset_archive(dataset_archive)
    verify_prepared_manifest_files()


def find_or_download_dataset_archive():
    data_settings = SETTINGS["data"]
    bundle = RESOLVED_PATHS["archive"]
    if data_settings["url"]:
        if not data_settings["url"].startswith("https://"):
            raise ValueError("Data URL must use HTTPS.")
        # Use a URL-specific cache name; changing URL does not reuse a different bundle.
        bundle = ROOT / (
            hashlib.sha256(data_settings["url"].encode()).hexdigest()[:16]
            + "-"
            + data_settings["archive"]
        )
        if not bundle.exists():
            partial = bundle.with_suffix(bundle.suffix + ".part")
            urllib.request.urlretrieve(data_settings["url"], partial)
            partial.replace(bundle)
    if not bundle.is_file():
        raise FileNotFoundError(f"Dataset archive not found: {bundle}")
    return bundle


def validate_and_extract_full_dataset_archive(bundle):
    data_settings = SETTINGS["data"]
    info = bundle.stat()
    identity = {
        "path": str(bundle.resolve()),
        "size": info.st_size,
        "modified_ns": info.st_mtime_ns,
        "manifests": data_settings["manifests"],
    }
    marker = RESOLVED_PATHS["marker"]
    try:
        previous = json.loads(marker.read_text()) if marker.exists() else None
    except (ValueError, OSError):
        previous = None
    if previous != identity:
        with zipfile.ZipFile(bundle) as archive:
            members = archive.infolist()
            required_manifests = set(data_settings["manifests"].values())
            if not required_manifests.issubset(archive.namelist()):
                raise ValueError(
                    f"Archive must contain prepared manifests: {sorted(required_manifests)}"
                )
            for archive_member_name in members:
                destination = (ROOT / archive_member_name.filename).resolve()
                if (
                    not destination.is_relative_to(ROOT.resolve())
                    or not archive_member_name.filename.startswith(
                        tuple(data_settings["allowed_archive_roots"])
                    )
                    or stat.S_ISLNK(archive_member_name.external_attr >> 16)
                ):
                    raise ValueError(
                        f"Unsafe or unexpected archive member: {archive_member_name.filename}"
                    )
            check_free_space(
                sum(archive_member_name.file_size for archive_member_name in members)
            )
            archive.extractall(ROOT)
        marker.parent.mkdir(parents=True, exist_ok=True)
        marker.write_text(json.dumps(identity), encoding="utf-8")


def verify_prepared_manifest_files():
    manifest_paths = [TRAINING_CONFIGURATION["manifest"]]
    if not TRAINING_CONFIGURATION.get("density_checkpoint"):
        manifest_paths.append(TRAINING_CONFIGURATION["city_manifest"])
    for path in manifest_paths:
        if not Path(path).is_file():
            raise FileNotFoundError(f"Missing extracted manifest: {path}")
    check_free_space()
    print("Prepared data:", RESOLVED_PATHS["data_root"])


def load_and_validate_dataset_annotations(configuration):
    if should_reuse_completed_training(configuration):
        return []
    if SETTINGS["action"] == ACTION_EVALUATE:
        options = SETTINGS["evaluation"]
        manifest = Path(configuration["output"]) / MATCHED_MANIFEST_FILE
        manifest.parent.mkdir(parents=True, exist_ok=True)
        from filelock import FileLock

        with FileLock(str(manifest) + ".lock", timeout=0):
            if not manifest.exists():
                cityscapes_manifest(
                    options["cityscapes_root"],
                    manifest,
                    with_detection_labels=True,
                    splits=("val",),
                )
        annotations = read_manifest(manifest, verify_images=True)
        root = Path(options["cityscapes_root"]).resolve()
        if any(not Path(row["image"]).is_relative_to(root) for row in annotations):
            raise ValueError(
                "cityscapes_root changed; choose a new evaluation output_name."
            )
        return annotations
    annotations = []
    manifest_keys = ["manifest"]
    if not configuration.get("density_checkpoint"):
        manifest_keys.append("city_manifest")
    for key in manifest_keys:
        path = Path(configuration[key])
        if not path.is_file():
            raise FileNotFoundError(f"Prepared manifest not found: {path}")
        if path.exists():
            print(f"Validating {path.name}")
            portable_rows = json.loads(path.read_text(encoding="utf-8"))
            for annotation in portable_rows:
                image_path = (path.parent / annotation["image"]).resolve()
                os.utime(image_path, ns=(946684800000000000, 946684800000000000))
            annotations.extend(read_manifest(path))
    return annotations


def display_dataset_split_summary(annotations, configuration):
    if not annotations:
        return
    frame = pd.DataFrame(annotations)
    display(
        frame.groupby(["dataset", "split"]).agg(
            images=("id", "size"),
            scenes=("group", "nunique"),
            known_beta=("beta", "count"),
        )
    )
    coverage_path = Path(configuration["manifest"]).with_suffix(".coverage.json")
    if coverage_path.exists():
        coverage = json.loads(coverage_path.read_text())
        display(
            pd.DataFrame(
                [
                    {
                        "expected": coverage["expected_images"],
                        "available": coverage["available_images"],
                        "missing": len(coverage["missing_images"]),
                        "complete": coverage["complete"],
                    }
                ]
            )
        )


def display_training_readiness_checks(configuration, annotations):
    if should_reuse_completed_training(configuration):
        checks = pd.DataFrame(
            [
                {
                    "check": "saved training complete",
                    "ready": True,
                    "detail": "Reuse completed/early-stopped comparisons; no training or inference",
                }
            ]
        )
        display(checks)
        return checks
    if SETTINGS["action"] == ACTION_EVALUATE:
        source = Path(configuration["evaluation_source"])
        source_rows = json.loads((source / INPUT_MANIFEST_FILE).read_text())
        selected = validate_matched_evaluation(
            annotations,
            source_rows,
            configuration["evaluation_split"],
            configuration.get("evaluation_max_pairs"),
        )
        checks = pd.DataFrame(
            [
                {
                    "check": "matched held-out images",
                    "ready": True,
                    "detail": f"{len(selected)} labeled images",
                }
            ]
        )
        display(checks)
        return checks
    checks = preflight(configuration, annotations)
    display(checks)
    if not checks.ready.all():
        display(checks.loc[~checks.ready, ["check", "detail"]])
    return checks


def validate_fog_data(configuration):
    """Load the selected annotations and validate every pre-training requirement."""
    annotations = (
        []
        if SETTINGS["action"] == ACTION_ANALYZE
        else load_and_validate_dataset_annotations(configuration)
    )
    display_dataset_split_summary(annotations, configuration)
    checks = (
        None
        if SETTINGS["action"] == ACTION_ANALYZE
        else display_training_readiness_checks(configuration, annotations)
    )
    return annotations, checks


def execute_analysis(configuration):
    return export_analytics(configuration["output"])


def execute_training(action, configuration, annotations, checks):
    if action == ACTION_PREFLIGHT:
        return pd.DataFrame(), None
    if action != ACTION_TRAIN:
        raise ValueError("Only action='train' can enter training.")
    if should_reuse_completed_training(configuration):
        print(
            "All comparisons complete. Reusing saved reports with their original settings."
        )
        results = execute_analysis(configuration)
        return results, pd.read_csv(Path(configuration["output"]) / STATUS_TABLE_FILE)
    if not checks.ready.all():
        print("Training is blocked by the checks above.")
        return pd.DataFrame(), None
    return run_training(annotations, configuration)


def train_or_analyze_configured_study(action, configuration, annotations, checks):
    action = str(action).strip().lower()
    if action not in {ACTION_TRAIN, ACTION_PREFLIGHT, ACTION_ANALYZE, ACTION_EVALUATE}:
        raise ValueError(
            "Choose train, preflight, analyze or evaluate; unknown actions never train."
        )
    print(f"Action: {action} | Output: {configuration['output']}")
    if action == ACTION_ANALYZE:
        return execute_analysis(configuration), None
    if action == ACTION_EVALUATE:
        return run_checkpoint_evaluation(annotations, configuration)
    return execute_training(action, configuration, annotations, checks)


def run_optional_synthetic_benchmark(enabled):
    """Measure a synthetic batch-64 ceiling without changing study results."""
    import time

    import torch
    import torchvision
    from torch.utils.data import DataLoader, TensorDataset

    can_run = enabled and torch.cuda.is_available()
    can_run = can_run and SETTINGS["profile"]["name"] == PROFILE_FULL
    if not can_run:
        print("Synthetic batch-64 benchmark skipped for this profile or CUDA state.")
        return
    print(f"Benchmarking on: {torch.cuda.get_device_name(0)}")
    image_count, batch_size = BENCHMARK_IMAGES, BENCHMARK_BATCH_SIZE
    images = torch.randn(image_count, 3, 640, 640, device="cuda", dtype=torch.float16)
    labels = torch.randn(image_count, 3, 640, 640, device="cuda", dtype=torch.float16)
    loader = DataLoader(TensorDataset(images, labels), batch_size=batch_size)
    model = torchvision.models.resnet50(weights=None).to("cuda").half()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(5):
        model(images[:batch_size])
    torch.cuda.synchronize()
    started = time.perf_counter()
    for batch_images, _ in loader:
        optimizer.zero_grad()
        model(batch_images).sum().backward()
        optimizer.step()
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    print(f"Processed {image_count} images in {elapsed:.2f} seconds.")
    print(f"Synthetic benchmark throughput: {image_count / elapsed:.2f} images/second.")


def display_accuracy_latency_and_density_tables(results, output):
    if results.empty:
        print("No completed runs yet.")
        return
    columns = [
        "dataset",
        "model",
        "arm",
        "seed",
        "density",
        "images",
        "mAP50_95",
        "AP50",
        "precision_at_025",
        "recall_at_025",
        "AR100",
    ]
    print(
        "Detection quality: higher is better. Clear = beta 0; foggy = pooled positive beta."
    )
    display(
        results.loc[
            results.density.isin(["all", "balanced", "0.0", "foggy"]), columns
        ].round(4)
    )
    print("Inference cost: lower milliseconds is better; higher FPS is better.")
    display(
        results.loc[
            results.density == "all",
            [
                "dataset",
                "model",
                "arm",
                "seed",
                "mean_ms",
                "p50_ms",
                "p95_ms",
                "serial_fps",
                "density_mean_ms",
            ],
        ].round(2)
    )
    paired = pd.read_csv(Path(output) / PAIRED_DELTA_FILE, dtype={"density": str})
    display(
        paired.loc[
            paired.density.isin(["all", "balanced"]),
            ["model", "dataset", "seed", "arm", "delta_mAP", "delta_latency_ms"],
        ].round(4)
    )
    density = pd.read_csv(Path(output) / DENSITY_REPORT_FILE)
    if not density.empty:
        display(density.round(6))
    hypothesis = pd.read_csv(Path(output) / HYPOTHESIS_FILE)
    if not hypothesis.empty:
        print("Positive delta means density helped detection; negative means it hurt.")
        display(hypothesis.round(4))
        summary = pd.read_csv(Path(output) / "hypothesis-summary.csv")
        display(summary.round(4))
    else:
        print(
            "Clear/fog hypothesis untested: this run has no labeled matched benchmark."
        )
    print(
        "Checkpoint health: best_epoch is the evaluated checkpoint; epochs_trained is total fitted epochs."
    )
    display(pd.read_csv(Path(output) / "training-health.csv"))
    print("Evaluation sample counts and labeled object counts:")
    display(pd.read_csv(Path(output) / "dataset-audit.csv"))


def display_saved_comparison_figures(output, results):
    if results.empty:
        return
    for pattern in (
        "*-comparison.png",
        "*-balanced.png",
        "*-deltas.png",
        "*-classes.png",
        "*-density.png",
        "*-loss.png",
        "*-conditions.png",
        "*-effects.png",
        "*-condition-classes.png",
        "*-precision-recall.png",
        "density-diagnostics.png",
        "dataset-audit.png",
    ):
        for path in sorted((Path(output) / FIGURE_DIRECTORY).glob(pattern)):
            display(PreviewImage(filename=str(path), width=1000))


def verify_subset_completion_and_checkpoint_reuse(results, annotations):
    """Check subset artifacts and reuse completed checkpoints without training again."""
    if should_reuse_completed_training(TRAINING_CONFIGURATION):
        print("Completed comparisons reused; no checkpoint loading or optimizer steps.")
        return
    # Fail the test if any stage or required checkpoint/report is missing.
    if (
        SETTINGS["profile"]["name"] != PROFILE_FULL
        and SETTINGS["action"] == ACTION_TRAIN
    ):
        required = [
            RESOLVED_PATHS["run"] / name
            for name in (
                RUN_SETTINGS_FILE,
                INPUT_MANIFEST_FILE,
                INPUT_SELECTION_FILE,
                RESULTS_FILE,
                PAIRED_DELTA_FILE,
                DENSITY_REPORT_FILE,
                SUBSET_AUDIT_FILE,
            )
        ]
        for seed in TRAINING_CONFIGURATION["seeds"]:
            has_density_targets = any(
                row["split"] == "density_train" and row.get("beta") is not None
                for row in annotations
            )
            if ARM_DENSITY in TRAINING_CONFIGURATION["arms"] and has_density_targets:
                required.extend(
                    RESOLVED_PATHS["run"] / f"{DENSITY_RUN_PREFIX}-{seed}" / name
                    for name in (LAST_CHECKPOINT, BEST_CHECKPOINT)
                )
            for model in TRAINING_CONFIGURATION["checkpoints"]:
                for arm in TRAINING_CONFIGURATION["arms"]:
                    directory = RESOLVED_PATHS["run"] / f"{model}-{arm}-{seed}"
                    required.extend(
                        directory / name
                        for name in (
                            LAST_CHECKPOINT,
                            BEST_CHECKPOINT,
                            STATUS_FILE,
                            RESULTS_FILE,
                        )
                    )
                    state = json.loads((directory / STATUS_FILE).read_text())
                    if state["status"] != "complete":
                        raise RuntimeError(f"Incomplete test run: {directory}")
        missing = [str(path) for path in required if not path.is_file()]
        if missing:
            raise RuntimeError(f"Missing test artifacts: {missing}")
        if results.empty:
            raise RuntimeError("Subset test produced no metrics.")
        import math

        overall = results.loc[results.density == "all"]
        for metric in ("mAP50_95", "AP50", "AR100", "mean_ms", "p95_ms"):
            if overall.empty or not overall[metric].map(math.isfinite).all():
                raise RuntimeError(
                    f"Undefined subset metric: {metric}. Check selected ground truth."
                )
        saved_times = {
            str(path): path.stat().st_mtime_ns
            for path in RESOLVED_PATHS["run"].glob(f"*/{LAST_CHECKPOINT}")
        }
        # Exercise completed-run resume; the engine should reuse checkpoints and saved results.
        resumed_results, resumed_status = run_training(
            annotations, TRAINING_CONFIGURATION, reset_setup=False
        )
        if any(
            Path(path).stat().st_mtime_ns != timestamp
            for path, timestamp in saved_times.items()
        ):
            raise RuntimeError(
                "Completed-run resume unexpectedly retrained checkpoints."
            )
        if resumed_results.empty or not resumed_status.status.eq("complete").all():
            raise RuntimeError("Completed-run resume failed.")
        print(
            "SUBSET E2E PASSED: training, evaluation, reports, checkpoints and completed-run resume."
        )


def export_checkpoints_and_reports():
    """Package the configured run and its recorded dependency versions."""
    if not RESOLVED_PATHS["run"].exists():
        raise FileNotFoundError(f"No run directory: {RESOLVED_PATHS['run']}")
    if RESOLVED_PATHS["packages"].exists():
        shutil.copy2(
            RESOLVED_PATHS["packages"],
            RESOLVED_PATHS["run"] / SETTINGS["output"]["packages_file"],
        )
    archive_path = shutil.make_archive(
        str(RESOLVED_PATHS["export"]), "zip", str(RESOLVED_PATHS["run"])
    )

    return archive_path

### Resolve storage and show the selected run

In [ ]:
os.environ["YOLO_CONFIG_DIR"] = str(ROOT / ".ultralytics")
os.environ["HF_HOME"] = str(ROOT / ".cache/huggingface")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

TRAINING_CONFIGURATION, RESOLVED_PATHS = resolve_study_configuration(SETTINGS)
display(
    pd.DataFrame(
        {
            "Setting": [
                "Profile",
                "Initialization",
                "Data source",
                "Output directory",
                "Models",
                "Arms",
                "Seeds",
                "Detector epochs",
                "Fog-density predictor",
                "Evaluation split",
                "Batch contract",
            ],
            "Value": [
                SETTINGS["profile"]["name"],
                SETTINGS["profile"]["initialization"],
                SETTINGS["data"]["location"],
                TRAINING_CONFIGURATION["output"],
                ", ".join(TRAINING_CONFIGURATION["checkpoints"]),
                ", ".join(TRAINING_CONFIGURATION["arms"]),
                TRAINING_CONFIGURATION["seeds"],
                TRAINING_CONFIGURATION["epochs"],
                TRAINING_CONFIGURATION.get("density_checkpoint")
                or "saved full-profile model",
                TRAINING_CONFIGURATION["evaluation_split"],
                f"micro={TRAINING_CONFIGURATION['batch_size']} / accumulate={TRAINING_CONFIGURATION['accumulate']} / effective={TRAINING_CONFIGURATION['effective_batch_size']}",
            ],
        }
    )
)
print(
    "Run behavior:",
    "reuse completed results"
    if should_reuse_completed_training(TRAINING_CONFIGURATION)
    else SETTINGS["action"],
)

In [ ]:
# fmt: off
#@title Fog-density model implementation (collapsed)
# fmt: on
class DensityCNN(nn.Module):
    """Small image-to-beta regressor used as the frozen fog predictor."""

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 5, 2, 2),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, 2, 1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(64, 1),
        )

    def forward(self, images):
        return (
            self.features(
                F.interpolate(
                    images,
                    (DENSITY_INPUT_SIZE, DENSITY_INPUT_SIZE),
                    mode="bilinear",
                    align_corners=False,
                )
            ).sigmoid()
            * BETA_SCALE
        )


def get_fog_density(images, predictor=None):
    """Estimate beta for a batch with the frozen image-to-density predictor."""
    predictor = predictor if predictor is not None else FOG_DENSITY_MODEL
    if predictor is None:
        raise ValueError("Run the fog-density model setup cell first.")
    with torch.inference_mode():
        return predictor(images)


def load_density_predictor(checkpoint_path, device):
    """Load once, switch to inference mode, and disable every parameter gradient."""
    global FOG_DENSITY_MODEL, FOG_DENSITY_MODEL_KEY
    checkpoint_path = Path(checkpoint_path).resolve()
    stamp = checkpoint_path.stat()
    key = (str(checkpoint_path), device, stamp.st_size, stamp.st_mtime_ns)
    if FOG_DENSITY_MODEL is not None and FOG_DENSITY_MODEL_KEY == key:
        return FOG_DENSITY_MODEL
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model = DensityCNN().to(device)
    model.load_state_dict(checkpoint["model"])
    FOG_DENSITY_MODEL = model.eval().requires_grad_(False)
    FOG_DENSITY_MODEL_KEY = key
    return FOG_DENSITY_MODEL


def setup_fog_density_model(configuration):
    """Restore the pipeline's saved predictor; report actual inference readiness."""
    reuse = should_reuse_completed_training(configuration)
    if SETTINGS["action"] == ACTION_ANALYZE or reuse:
        print("Saved reports selected; model loading is skipped.")
        return None
    if ARM_DENSITY not in configuration["arms"]:
        print("RGB-only comparison; no fog predictor is needed.")
        return None
    checkpoint = configuration.get("density_checkpoint")
    if SETTINGS["action"] == ACTION_EVALUATE:
        source = Path(configuration["evaluation_source"])
        saved = json.loads((source / RUN_SETTINGS_FILE).read_text())["config"]
        checkpoint = (
            saved.get("density_checkpoint")
            or source
            / f"{DENSITY_RUN_PREFIX}-{SETTINGS['density_predictor']['seed']}"
            / BEST_CHECKPOINT
        )
    if not checkpoint:
        seed = SETTINGS["density_predictor"]["seed"]
        source = configuration.get("evaluation_source", configuration["output"])
        checkpoint = Path(source) / f"{DENSITY_RUN_PREFIX}-{seed}" / BEST_CHECKPOINT
    checkpoint = Path(checkpoint)
    if not checkpoint.is_file():
        if configuration.get("profile_name") == PROFILE_FULL and not configuration.get(
            "density_checkpoint"
        ):
            print(
                "Existing full-run predictor will resume from its saved per-seed checkpoint before detector training."
            )
            return None
        raise FileNotFoundError(
            f"Fog predictor not found: {checkpoint}. Set density_predictor.checkpoint to your saved {BEST_CHECKPOINT}."
        )
    model = load_density_predictor(checkpoint, configuration["device"])
    print(f"Loaded fog predictor: {checkpoint}")
    frozen = all(not parameter.requires_grad for parameter in model.parameters())
    print(f"Frozen: {frozen} | eval: {not model.training} | {configuration['device']}")
    return model

### Load the fog-density model (pretrained, frozen)

This step opens the saved checkpoint, loads `DensityCNN`, switches it to evaluation mode, and disables parameter gradients. The same loaded predictor supplies beta through `get_fog_density(images)`. Its printed status confirms the actual path, device, and frozen state. Saved-report mode skips model loading. Existing full runs retain their per-seed checkpoint continuation.

In [ ]:
fog_predictor = setup_fog_density_model(TRAINING_CONFIGURATION)

## 3. Prepare the study data

Prepare the profile-specific nuScenes images and labeled boxes. The frozen fog-density predictor is already selected above; the notebook does not stage a second fog-density training dataset.

In [ ]:
prepare_fog_data()

## 4. Validate the fog data

Check image and box annotations, the saved beta predictor, split boundaries, and the selected run before spending GPU time.

In [ ]:
rows, checks = validate_fog_data(TRAINING_CONFIGURATION)

## 5. Train and evaluate the detectors

Train or resume the configured object detectors and their matched RGB/predicted-density arms. The density model remains frozen; its beta estimate is supplied through `get_fog_density()` inside the detection pipeline.

### Training progress

The detector models show cumulative epoch and image-step progress. The pretrained beta estimator is a frozen input predictor; only the object detectors and their matched adapters train in new partial profiles.

In [ ]:
if SETTINGS["action"] in {"train", "preflight"}:
    display(read_run_continuation_summary(TRAINING_CONFIGURATION))

In [ ]:
results, status = train_or_analyze_configured_study(
    SETTINGS["action"], TRAINING_CONFIGURATION, rows, checks
)
if status is not None:
    display(status)

In [ ]:
RUN_SYNTHETIC_BENCHMARK = False
run_optional_synthetic_benchmark(RUN_SYNTHETIC_BENCHMARK)

## 6. Review the outputs

Detector results compare the RGB control with the frozen beta-predictor arm, including clear/fog accuracy, fog-level effects, class results, and latency.

### Tables: accuracy, overhead, and the hypothesis

`fog-vs-clear.csv` records each matched training seed's clear delta, fog delta and fog advantage (fog delta minus clear delta). `hypothesis-summary.csv` reports means and seed spread; at least three independent training seeds are required for exploratory bootstrap intervals. A missing interval is not a finding of no effect. The old nuScenes results stay separate with their original evaluation split.

In [ ]:
display_accuracy_latency_and_density_tables(results, TRAINING_CONFIGURATION["output"])

### Detector-study graphs

These figures summarize the object-detection comparison, including accuracy, fog-level behavior, class results, and latency with predictor cost included.

- **Clear/fog bars and effect bars:** compare each model and arm; zero is no effect.
- **Fog-level curves:** see whether a benefit changes with density.
- **Class heatmaps:** compare object categories; blank cells have no labeled examples.
- **Accuracy versus latency:** compare detection quality and inference cost.
- **Density diagnostics:** compare estimated beta with known beta.
- **Training loss and data audit:** check optimization progress and sample coverage.

In [ ]:
display_saved_comparison_figures(TRAINING_CONFIGURATION["output"], results)

### Subset completion and checkpoint reuse

In [ ]:
verify_subset_completion_and_checkpoint_reuse(results, rows)

## 7. Export the run

The ZIP is saved beside the run directory in the configured output location. It includes reports, checkpoints and the installed package versions.


In [ ]:
archive_path = export_checkpoints_and_reports()
print("Saved:", archive_path)

### Interpret the comparison

The predicted-density arm uses the frozen beta estimate as a detector input. The RGB control receives zero beta. Compare their labeled clear and fog results to assess whether that input changes object-detection performance.

Results persist in the selected output folder. Use `analyze` to redraw saved detector reports, or `evaluate` to resume matched predictions from saved detector weights. The export cell packages the active output folder.